# Paper 5, RB-9: position erased from the residual stream before attention and before the MLP of every block, encoding on

**Set on 10 Nov 2026 (IST), before the run.** Question raised after RB-8 (group, 10 Nov): RB-8 edits each block's input only
(a forward pre-hook, before attention). Inside every block, attention writes position back into the residual stream, and that
block's MLP reads it before the next block's edit removes it. RB-9 closes that gap.

**Sites.** The same edit as RB-8 (`mean`: subtract each patch slot's positional component c(p) = mu(p) - mu_bar, fitted under
RPI), now applied to the residual stream at every point a later computation reads it:

* `attn b`: entering block b, before its attention (RB-8's site).
* `mlp b`: the residual stream after block b's attention has been added, before its MLP. The stream itself is edited, not
  only the MLP input: the MLP reads the edited stream, and the block's output carries the edited stream forward.
* `out`: the output of the last block, before the final norm.

Pure RoPE models: `mlp 0`, then `attn b` and `mlp b` for blocks 1-11, then `out` (24 sites; the input of block 0 has no
positional component, RB-6 Part A). Hybrid and APE: `attn b` and `mlp b` for blocks 0-11, then `out` (25 sites).
Prefix tokens (class / register) are never edited, as in RB-6 and RB-8. Each site is fitted **in order with every earlier
site already edited**, on 16,384 images taken right after the evaluation images (as RB-8's `mean`).

**Arms per model.** intact; floor(s); `rb8_mean` (RB-8's all-block edit rebuilt from RB-8's own fit files, must equal RB-8 bit
for bit); `full_mean` (all sites); `full_offset` (control: the same vectors subtracted at the wrong slots, same sizes, no
position removed). Probes at every edited site under `full_mean`.

**Reading, fixed before the run** (RB-8's rule, unchanged): per classifier, C = top-1 cost of `full_mean`, C0 = cost of
`full_offset`, F = cost of removing position entirely (rotations off; embedding zeroed; both in the hybrid).
BARELY USED if C < 10% of F; USED if C >= 10% of F and C - C0 > 3 paired SE; CANNOT TELL otherwise.
The statement covers the per-slot mean component only.

**What remains outside the edit, by design.** Position used *inside* attention (RoPE rotates q and k before the dot
product; that is the mechanism the paper describes, not something stored in the stream), and the class / register tokens.


In [ ]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas", "transformers"]:
    ensure(p)
print("environment ready")

In [ ]:
import os, json, time, warnings, traceback, shutil, hashlib, platform, socket
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.executable)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# config
REPO_PATH  = os.environ.get("RB9_REPO", "")
RB0_DIR    = os.environ.get("RB9_RB0_DIR", "")
RB8_DIR    = os.environ.get("RB9_RB8_DIR", "")       # RB-8's output folder with its fit files (rb8_fit_<model>_<rep|acc>_mean.npz)
MODELS_RUN = [m for m in os.environ.get("RB9_MODELS",
              "naver,small,reg1_gap,rope_ape,DINOv3,supervised,augreg,sam,deit3,dino").split(",") if m]   # RoPE + hybrid first
RESUME     = os.environ.get("RB9_RESUME", "1") == "1"
STRICT     = os.environ.get("RB9_STRICT", "1") == "1"
TEST_MODE  = os.environ.get("RB9_TEST", "0") == "1"   # CPU smoke test against a test-mode RB-8 output
OUT_DIR    = Path(os.environ.get("RB9_OUT", "rb9_output"))

N_SSDC, N_ACC, BATCH_SIZE = 1000, 5000, 256
N_FIT_MEAN = 16384                              # as RB-8's mean edit, both datasets
METRIC, TORCH_SEED, INPUT_SIZE = "manhattan", 20260908, 224
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"
ACC_DATASET  = "ILSVRC/imagenet-1k"
N_BLOCKS, FINAL = 12, 12
MLP0 = 13                                       # readout 13 + b = block b's MLP input (after norm2), on the edited stream
PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8   # W2-W6 / RB-0 / RB-2 / RB-8 protocol, unchanged
PROBE_CHUNK, PROBE_DEVICE = 8192, None
MAX_CAPTURE = 13                                # readouts captured per forward sweep (RB-8's maximum; ~8 GB CPU RAM for ViT-B)

SITE_MAIN = 4
SUBSPACE_ENERGY = 0.99
SUBSPACE_SEED, SCRAMBLE_SEED = 20261108, 7      # RB-6 / RB-8 values (derangement of the offset control)
NORM_MIN_RANGE = 0.10
ERASE_N_MAX, ERASE_EXACT_X, ERASE_AXIS_X = 0.10, 4.0, 1.5
FLAT_TOL = 1e-6
DISJOINT_MAX_OVERLAP = 5
STRUCT_TOL = 1e-5                               # hooked block vs reference block, relative to max |output| (float add order only)

EXACT_TOL = 0.0
W1B_TOL_SSDC = 1e-3
W7_TOL_POINTS, W7_MIN_AGREE = 0.10, 0.995
EXPECT_TORCH, EXPECT_CUDNN = "2.10.0+cu128", 91002
EXPECT_CUDNN_PKGS = {"nvidia-cudnn-cu12": "9.10.2.21"}
FIRST_LAYER_910 = {"naver": 0.00041303808169361526, "DINOv3": 0.0004745740108653513,
                   "small": 0.000510305810057617, "reg1_gap": 0.00026782853342761795}
FIRST_LAYER_RANGE = (1e-4, 2e-3)

MODEL_SPECS = {
    "naver":      dict(family="rope",   loader="rope", name="vit_base_patch16_rope_224.naver_in1k",        classifier=True,  label="naver ViT-B"),
    "DINOv3":     dict(family="rope",   loader="rope", name="vit_base_patch16_dinov3.lvd1689m",            classifier=False, label="DINOv3 ViT-B"),
    "small":      dict(family="rope",   loader="rope", name="vit_small_patch16_rope_224.naver_in1k",       classifier=True,  label="naver ViT-S"),
    "reg1_gap":   dict(family="rope",   loader="rope", name="vit_base_patch16_rope_reg1_gap_256.sbb_in1k", classifier=True,  label="reg1-gap ViT-B"),
    "rope_ape":   dict(family="hybrid", loader="rope", name="vit_base_patch16_rope_ape_224.naver_in1k",    classifier=True,  label="hybrid ViT-B"),
    "supervised": dict(family="ape",    loader="hf",   name="google/vit-base-patch16-224",                 classifier=True,  label="supervised ViT-B"),
    "dino":       dict(family="ape",    loader="timm", name="vit_base_patch16_224.dino",                   classifier=False, label="DINO ViT-B"),
    "augreg":     dict(family="ape",    loader="timm", name="vit_base_patch16_224.augreg_in1k",            classifier=True,  label="AugReg ViT-B"),
    "sam":        dict(family="ape",    loader="timm", name="vit_base_patch16_224.sam_in1k",               classifier=True,  label="SAM ViT-B"),
    "deit3":      dict(family="ape",    loader="timm", name="deit3_base_patch16_224.fb_in1k",              classifier=True,  label="DeiT-III ViT-B"),
}
def has_rope(key): return MODEL_SPECS[key]["family"] in ("rope", "hybrid")
def floor_name(key): return "both_floor" if MODEL_SPECS[key]["family"] == "hybrid" else "floor"

def sites_of(key):
    """Edit sites in forward order: ('attn', b) entering block b; ('mlp', b) the stream before block b's MLP; ('out', 11)."""
    first = 1 if MODEL_SPECS[key]["family"] == "rope" else 0
    s = [] if first == 0 else [("mlp", 0)]
    for b in range(first, N_BLOCKS):
        s += [("attn", b), ("mlp", b)]
    return s + [("out", N_BLOCKS - 1)]
def site_name(s): return f"{s[0]}{s[1]}" if s[0] != "out" else "out"
def site_readout(s): return s[1] if s[0] == "attn" else (MLP0 + s[1] if s[0] == "mlp" else FINAL)
def rb8_blocks(key): return list(range(1 if MODEL_SPECS[key]["family"] == "rope" else 0, N_BLOCKS))   # RB-8's edited blocks

RB0_SHA256 = {
    "w8_partA_naver.json":     "f50b694d24279ce63b8dac8da1906b7e60ab6522ce8de4c992b1c87ceed10914",
    "w8_partA_DINOv3.json":    "f1d963652ac3867c79bb170da2b2dd86ff8e978008f5ff59304a69947bb909b5",
    "w8_partA_small.json":     "be01ed6297ef819a8fc219853c07fc9a81a76d2c6ffbe9da8501379fdf6acbbb",
    "w8_partA_reg1_gap.json":  "da5ae49a2abc21aa0df8f4c43b249727a7edfe444254e239e637ed96a312fefb",
    "w8_partA_rope_ape.json":  "679d30cbd1b8873d3aea6724f2fa46984dbd5308f167ff0711f51bcdd6150c92",
    "w8_partC_naver.json":     "c983414404ca645bc8144a2bd4ec76bfb583f4c5c9588ec8faffe853e5c3b985",
    "w8_partC_small.json":     "bf895274486dfaa9ab042118e15bb7c9863a5f6201ab27ec11fb539ecf7126f6",
    "w8_partC_reg1_gap.json":  "8b5641235e05b20bfdc597032097b2f85878c228bdc32ec8b63888227e393698",
    "w8_partC_rope_ape.json":  "c0d4b9fec550b3343b52f5f267ceb277b1dcfb81b59efc1d0805734b234bb2bb",
    "w8_perimage_naver.npz":   "5fe4a53ecbf99bc583c56ef6ec9f774c5bd6027e1734dce16fe0c230c36358c6",
    "w8_perimage_small.npz":   "b9f6a3833766b73618c82b3e5ecbb81af672b11eaa49fab503e27bb8734cf40b",
    "w8_perimage_reg1_gap.npz": "10310f0c0109d65f0b594f7774006ddd2c69f5fc74438b13d32956d39738643e",
    "w8_perimage_rope_ape.npz": "73bae58e5d02d1035b597ec1c6e0c613b37a582b8bb2e67b515d9471ddbba192",
}
RB0_FILES = {k: [f"w8_partA_{k}.json"] + ([f"w8_partC_{k}.json", f"w8_perimage_{k}.npz"] if s["classifier"] else [])
             for k, s in MODEL_SPECS.items() if s["family"] != "ape"}
W1B_FILE, W1B_SHA256 = "w1b/w1b_floors_2026-09-09T07-55-22Z.json", "cd776ee5f8e62d9263f5fcb2a583d2aa34a3ecf3aab6ca75703b38df90365669"
W1B_KEY = {"supervised": "APE · google ViT", "dino": "APE · DINOv1"}
W7_APE = {
    "supervised": ("w7/w7ape_supervised_2026-09-24T18-12-56Z.json", "18dd342d8b9d2d9f154e87e84062671ddefc833a8dc240b0edf4837dab98555d",
                   "w7/perimage/ape_supervised_correct.npz", "fcd2ee3849f3a47247d346c232d3de61e1596ba4564144c15dcdb075aac66f95"),
    "augreg":     ("w7/w7ape_augreg_2026-09-24T18-12-56Z.json", "06e946e04cc97de6fa79822475e24604e451d2d94301014e867df602a35de50e",
                   "w7/perimage/ape_augreg_correct.npz", "fb877584e1d1c6688b5736ef775ce75563e84655fc0c145bcf41b8f36378b688"),
    "sam":        ("w7/w7ape_sam_MERGED.json", "f8d754fac93bffb10e4652cd712a339637ae658ad4471d49b2603dd0c348dc65",
                   "w7/perimage/ape_sam_correct.npz", "c2e83a69afa8b6005d340a760ee735ad72ca967d35b0a3d046cb3b6f5f498dff"),
    "deit3":      ("w7/w7ape_deit3_2026-09-25T01-02-08Z.json", "0d42773eb22dc2dd4f2bede943eac73d99130f55e6f859a39144fdac557c6f87",
                   "w7/perimage/ape_deit3_correct.npz", "c851a1258715a6476e5d7f9a00e7aa2573266fb29329800ac1edbe1b486ad764"),
}
REPLAY_CHECK_N = 512

# RB-8's result files (rb8_output.zip, 9 Nov); fit files are checked against the sha256 RB-8 recorded
RB8_SHA256 = {
    "rb8_naver.json": "955758b94ccaa331de9a359a3bed472f371a8d94d1ee466ea144edb520745f6d",
    "rb8_DINOv3.json": "e5f9576da1403b9ab55b540309f29465357f4ad3af4e2086c4306e20aab9ca8b",
    "rb8_small.json": "bcd83fe4faad6f195a687f1051281bcb9af4dfbf1d6f1692da6c0dbaa8435ac8",
    "rb8_reg1_gap.json": "2c5dae5a29ae8506abc5cd72bb2de37f98e9b9148e53dd88c806b103c3d43e3d",
    "rb8_rope_ape.json": "60df27f2d54b49381c0e9e9557d87b19286fe53eb0696db8fe2d206685687fb9",
    "rb8_supervised.json": "45a5df6d2310be6f8482c53e8b8539a5542d093980a643aab4f080de64af852a",
    "rb8_dino.json": "6ea27cf31aa8f6b3618c4a7bf3d43379af6bf4ea0cbb4bcf27399f9ac88c3e52",
    "rb8_augreg.json": "adee9ed55ba986da73571ef961d0a439c8394ab190130c91360d11cce9251315",
    "rb8_sam.json": "2c5c9cb45076ccd8f712e51c3c010183e338923d704400a6fbb7249eb64ae239",
    "rb8_deit3.json": "fd64de5a39c68a7a24b87fcc00f8847135002dfa1dd8b6e5ef36b072d5a1fd5d",
    "rb8_perimage_naver.npz": "6961b15b820d77c4ec9af4b8f9fa957f331a109d207d0687e1dd75f1f6204d1a",
    "rb8_perimage_small.npz": "e39a622e1471d284e8b9f4fb836f0787c1fae5b72560584e73de0c5881099bcd",
    "rb8_perimage_reg1_gap.npz": "2778638dc98631e65c8ff8abf2b3b6b7195690f318b28eaa1d206f08aecc0691",
    "rb8_perimage_rope_ape.npz": "ff0e96e60c638486397c33b6de0bc24c2c222dca196e472835f96b1f06997848",
    "rb8_perimage_supervised.npz": "6b7227051222f1e1931169747835ee0c18901ff759702fcbde20098b658c897d",
    "rb8_perimage_augreg.npz": "00efba18dabc32236aee8ff5668c76a4e491745a5332d6557da4a288d2900baa",
    "rb8_perimage_sam.npz": "3a407b4a8ec364f4bf77cfc59c1e30cd832ad093f20cb66fe80338d6130bca55",
    "rb8_perimage_deit3.npz": "568095cb07c45a83b5fdc70a249b2cd46d0e18d974ac79a7e0c8291dc3cb2589",
}

RULE_ERASE = (f"Erasure check at an edited site (its readout sees the edited stream): normalised SSDC N <= {ERASE_N_MAX}, and on held-out images "
              f"exact probe <= {ERASE_EXACT_X}x chance and row and column probes <= {ERASE_AXIS_X}x chance (RB-6 / RB-8 thresholds). "
              "Where N is undefined (intact - floor < 0.10) the probes decide.")
RULE_NORM = (f"N = (SSDC - floor) / (intact - floor) at the same readout, only where intact - floor >= {NORM_MIN_RANGE}. "
             "Floor = all rotations off (RoPE), embedding zeroed (APE), both (hybrid).")
RULE_TOP1 = "Top-1 cost = intact - arm in points on the same 5,000 images, with a paired SE."
READ_SHARE, READ_SE = 0.10, 3.0
RULE_READING = (f"Classifiers (RB-8's rule, unchanged). C = top-1 cost of full_mean (every site), C0 = cost of full_offset (the same vectors at the "
                f"wrong slots), F = cost of removing position entirely. BARELY USED if C < {READ_SHARE:.0%} of F; USED if C >= {READ_SHARE:.0%} "
                f"of F and C - C0 > {READ_SE:.0f} paired SE; CANNOT TELL otherwise. The statement covers the per-slot mean component only.")
RULE_OUTCOME = ("Outcome (labels only; the team decides the text): A if every RoPE classifier (naver ViT-B, naver ViT-S, reg1-gap) reads BARELY USED; "
                "B if any reads USED; C otherwise. The hybrid and the APE models are reported with the same rule.")

if TEST_MODE:
    N_SSDC, N_ACC, BATCH_SIZE = 24, 32, 8
    N_FIT_MEAN = 80
    PROBE_PASSES = 2
    OUT_DIR = Path(os.environ.get("RB9_OUT", "rb9_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)
_bad = [m for m in MODELS_RUN if m not in MODEL_SPECS]
if _bad: raise SystemExit(f"Unknown entries in RB9_MODELS: {_bad}")

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"rb9_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return bool(passed) or skip
print(f"device {DEVICE} | fp32 | models {MODELS_RUN}")
print(f"SSDC/probes: {SSDC_DATASET} x {N_SSDC} | top-1: {ACC_DATASET} x {N_ACC} | fit: {N_FIT_MEAN} per dataset | batch {BATCH_SIZE} | seed {TORCH_SEED}")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | strict {STRICT} | test mode {TEST_MODE} | run {RUN_STAMP}")


## Pre-flight: are we in `env_paper5_cudnn910`?
Same checks as RB-2 to RB-5. Nothing is measured if any fails.

In [ ]:
def _sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as ex:
        return f"(failed: {ex!r})"

def _loaded_cudnn():
    try:
        with open("/proc/self/maps") as f:
            return sorted({l.split()[-1] for l in f if "libcudnn" in l and l.split()[-1].startswith("/")})
    except Exception:
        return []

import importlib.metadata as _md
if DEVICE == "cuda":
    _x = torch.randn(2, 3, 32, 32, device="cuda"); _w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(_x, _w, stride=16); torch.cuda.synchronize(); del _x, _w
CUDNN_PKGS = {}
for _d in _md.distributions():
    _n = (_d.metadata.get("Name") or "").lower()
    if "cudnn" in _n: CUDNN_PKGS[_n] = _d.version
import transformers as _tf
ENV = {
    "python": sys.executable, "prefix": sys.prefix, "host": socket.gethostname(), "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "transformers": _tf.__version__, "numpy": np.__version__,
    "cudnn_reported": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "libcudnn_loaded": _loaded_cudnn(), "cudnn_packages": CUDNN_PKGS,
    "flags": {"matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
              "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
              "float32_matmul_precision": torch.get_float32_matmul_precision()},
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu": _sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv") if DEVICE == "cuda" else "cpu",
}
(OUT_DIR / f"rb9_pip_freeze_{RUN_STAMP}.txt").write_text(_sh(f"{sys.executable} -m pip freeze", timeout=300))
(OUT_DIR / f"rb9_environment_{RUN_STAMP}.json").write_text(json.dumps(ENV, indent=1, default=str))
print(json.dumps({k: v for k, v in ENV.items() if k != "gpu"}, indent=1, default=str)); print(ENV["gpu"])

_pre = []
if TEST_MODE:
    gate("pre-flight (cuDNN 9.10.2 environment)", "env", True, "test mode on CPU", skip=True)
else:
    _pre.append(gate("CUDA available", "env", DEVICE == "cuda"))
    _pre.append(gate(f"torch is {EXPECT_TORCH}", "env", torch.__version__ == EXPECT_TORCH, torch.__version__))
    _pre.append(gate(f"PyTorch reports cuDNN {EXPECT_CUDNN} (9.10.2)", "env", ENV["cudnn_reported"] == EXPECT_CUDNN,
                     str(ENV["cudnn_reported"])))
    _prefix = os.path.realpath(sys.prefix)
    _libs = ENV["libcudnn_loaded"]
    _pre.append(gate("every loaded libcudnn file comes from this environment's nvidia/cudnn folder", "env",
                     bool(_libs) and all(os.path.realpath(l).startswith(_prefix) and "/nvidia/cudnn/" in l for l in _libs),
                     "; ".join(Path(l).name for l in _libs) + f" (prefix {_prefix})"))
    _pre.append(gate("the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13)", "env",
                     CUDNN_PKGS == EXPECT_CUDNN_PKGS, str(CUDNN_PKGS)))
    _f = ENV["flags"]
    _pre.append(gate("TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0)", "env",
                     _f["matmul_allow_tf32"] is False and _f["cudnn_allow_tf32"] is True and _f["cudnn_benchmark"] is False
                     and _f["cudnn_deterministic"] is False and _f["float32_matmul_precision"] == "highest", str(_f)))
PREFLIGHT_OK = all(_pre)
if not PREFLIGHT_OK:
    raise SystemExit("PRE-FLIGHT FAILED: this notebook must run in env_paper5_cudnn910 (cuDNN 9.10.2), with PyTorch's "
                     "default flags. Nothing was measured. Switch the kernel / python and run again.")
print("\npre-flight OK")

In [ ]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB9_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
REPO_COMMIT = _sh(f"git -C {repo} rev-parse HEAD")
print("repo root:", repo, "| commit", REPO_COMMIT)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope, load_ape, load_ape_timm
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

## The canonical RB-0, the APE anchors and RB-8
RoPE and hybrid intact / floor arms are gated against RB-0 parts A and C (the cuDNN 9.10.2 re-run), checked by sha256; APE
models against W1b (SSDC) and W7 (top-1) within tolerance, as in RB-8. RB-8's result files are checked by sha256, and its fit files
against the sha256 RB-8 recorded for them; RB-8's all-block edit is then rebuilt from those fits and must equal RB-8 bit for bit.

In [ ]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

RB0 = {}
_need_rb0 = [k for k in MODELS_RUN if k in RB0_FILES]
if _need_rb0 and not RB0_DIR and not TEST_MODE:
    raise SystemExit("Set RB9_RB0_DIR to the canonical RB-0 folder: "
                     "/workspace/NRDL/experiments/rb0_cudnn910/w8_output")
_rb0 = Path(RB0_DIR).expanduser() if RB0_DIR else None
_rb0_bad = []
for key in _need_rb0:
    for fn in RB0_FILES[key]:
        p = (_rb0 / fn) if _rb0 else None
        if p is None or not p.exists():
            if TEST_MODE: gate(f"RB-0 file {fn} present", key, True, "test mode: no RB-0 folder", skip=True)
            else: gate(f"RB-0 file {fn} present", key, False, str(p)); _rb0_bad.append(fn)
            continue
        if TEST_MODE:
            gate(f"{fn} is the canonical RB-0 file", key, True, "test mode: hash not checked", skip=True)
        else:
            d = sha256_file(p)
            if not gate(f"{fn} is the canonical RB-0 file (cuDNN 9.10.2 re-run)", key, d == RB0_SHA256[fn],
                        f"sha256 {d[:16]}... (expected {RB0_SHA256[fn][:16]}...)"):
                _rb0_bad.append(fn)
if _rb0_bad:
    raise SystemExit(f"RB-0 FINGERPRINT FAILED for {_rb0_bad}. RB8_RB0_DIR must be the cuDNN 9.10.2 re-run "
                     "(rb0_cudnn910/w8_output, 4 Nov). Nothing was measured.")
for key in _need_rb0:
    d = {}
    for part in "AC":
        p = (_rb0 / f"w8_part{part}_{key}.json") if _rb0 else None
        d[part] = json.loads(p.read_text()) if (p is not None and p.exists()) else None
    p = (_rb0 / f"w8_perimage_{key}.npz") if _rb0 else None
    d["P"] = ({n: z_[n] for z_ in [np.load(p)] for n in z_.files} if (p is not None and p.exists()) else None)
    RB0[key] = d
    need = "AC" if MODEL_SPECS[key]["classifier"] else "A"
    gate("RB-0 parts needed by RB-8 are present and complete", key,
         all((d[x] or {}).get("complete") for x in need) and (d["P"] is not None or not MODEL_SPECS[key]["classifier"]),
         ", ".join(f"{x}={'complete' if (d[x] or {}).get('complete') else 'missing'}" for x in need))

RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
W1B, W7A = {}, {}
_ape_run = [k for k in MODELS_RUN if MODEL_SPECS[k]["family"] == "ape"]
if TEST_MODE:
    if _ape_run: gate("committed W1b / W7 APE anchors loaded", "anchors", True, "test mode: tiny random models have no anchors", skip=True)
elif _ape_run:
    f1 = RES_ROOT / W1B_FILE
    if any(k in W1B_KEY for k in _ape_run):
        if gate("committed W1b file found and is the expected one (sha256)", "W1b", f1.exists() and sha256_file(f1) == W1B_SHA256,
                str(f1) + ("" if not f1.exists() else f" sha256 {sha256_file(f1)[:16]}...")):
            e1 = json.load(open(f1))["results"]
            for k, k1 in W1B_KEY.items():
                if k in _ape_run:
                    W1B[k] = {"baseline": e1[k1]["baseline_ssdc"], "floor": e1[k1]["floor_ssdc"]}
    for k in _ape_run:
        if k not in W7_APE: continue
        fj, hj, fi, hi = W7_APE[k]; pj, pi = RES_ROOT / fj, RES_ROOT / fi
        ok = pj.exists() and pi.exists() and sha256_file(pj) == hj and sha256_file(pi) == hi
        if gate("committed W7 APE files (results + per-image) found and are the expected ones (sha256)", k, ok, f"{fj}; {fi}"):
            W7A[k] = {"acc": float(json.load(open(pj))["conditions"]["intact"]["acc"]), "img": np.load(pi)["intact"].astype(np.uint8)}
print("W1b anchors for:", list(W1B), "| W7 top-1 anchors for:", list(W7A))

# RB-8 results used as anchors
RB8 = {}
if not RB8_DIR:
    raise SystemExit("Set RB9_RB8_DIR to RB-8's output folder, the one that still holds the fit files rb8_fit_<model>_<rep|acc>_mean.npz.")
_rb8 = Path(RB8_DIR).expanduser()
_rb8_bad = []
for key in MODELS_RUN:
    names = [f"rb8_{key}.json"] + ([f"rb8_perimage_{key}.npz"] if MODEL_SPECS[key]["classifier"] else [])
    for fn in names:
        p = _rb8 / fn
        if not p.exists():
            gate(f"RB-8 file {fn} present", key, False, str(p)); _rb8_bad.append(fn); continue
        if TEST_MODE:
            gate(f"{fn} is the expected RB-8 file", key, True, "test mode: hash not checked", skip=True)
        else:
            d = sha256_file(p)
            if not gate(f"{fn} is the RB-8 file of 9 Nov (sha256)", key, d == RB8_SHA256[fn], f"{d[:16]}..."): _rb8_bad.append(fn)
    if f"rb8_{key}.json" in _rb8_bad: continue
    R8 = json.loads((_rb8 / f"rb8_{key}.json").read_text())
    for t in ["rep"] + (["acc"] if MODEL_SPECS[key]["classifier"] else []):
        fn = f"rb8_fit_{key}_{t}_mean.npz"; p = _rb8 / fn
        if not p.exists():
            gate(f"RB-8 fit file {fn} present (kept out of the zip; it is in the run folder)", key, False, str(p)); _rb8_bad.append(fn); continue
        d = sha256_file(p)
        if not gate(f"{fn} is the fit RB-8 was built from (sha256 recorded in rb8_{key}.json)", key,
                    d == R8.get("fit_sha256", {}).get(f"{t}_mean"), f"{d[:16]}..."): _rb8_bad.append(fn)
if _rb8_bad:
    raise SystemExit(f"RB-8 FILES MISSING OR CHANGED: {_rb8_bad}. Nothing was measured.")
for key in MODELS_RUN:
    R8 = json.loads((_rb8 / f"rb8_{key}.json").read_text())
    RB8[key] = {"R": R8, "fit": {t: {n: z_[n] for z_ in [np.load(_rb8 / f"rb8_fit_{key}_{t}_mean.npz")] for n in z_.files}
                                 for t in ["rep"] + (["acc"] if MODEL_SPECS[key]["classifier"] else [])}}
    if MODEL_SPECS[key]["classifier"]:
        z = np.load(_rb8 / f"rb8_perimage_{key}.npz"); RB8[key]["P"] = {n: z[n] for n in z.files}
    gate("RB-8 result for this model is complete", key, bool(R8.get("complete")) and "all_mean" in R8["ssdc"]
         and (not MODEL_SPECS[key]["classifier"] or "all_mean" in R8["acc"]))


In [ ]:
# datasets: explicit ids, no fallback; fitting images follow the evaluation images
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed, start=0):
            self.n, self.seed, self.start = n, seed, start
        def skip(self, k): return _FakeStream(self.n, self.seed, self.start + k)
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                if i >= self.start: yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1); DS_ACC = _FakeStream(400, 2)
    print("TEST MODE: random images stand in for both datasets")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    DS_ACC = load_imagenet(split="validation", streaming=True, dataset_id=ACC_DATASET, fallback_id=None)
    gate("SSDC dataset is the repack (W2-W6, RB-0 to RB-5)", "data", SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)
    gate("top-1 dataset is the ILSVRC split (W7, RB-0 to RB-5)", "data", ACC_DATASET == "ILSVRC/imagenet-1k", ACC_DATASET)
FIT_SRC = {"rep": (lambda: DS_SSDC.skip(N_SSDC), f"{SSDC_DATASET} images {N_SSDC}..{N_SSDC + N_FIT_MEAN - 1}"),
           "acc": (lambda: DS_ACC.skip(N_ACC), f"{ACC_DATASET} images {N_ACC}..{N_ACC + N_FIT_MEAN - 1}")}

In [ ]:
# source-aware helpers (timm and transformers)
def pix(im):  return im if torch.is_tensor(im) else im["pixel_values"]
def to_dev(im):
    if torch.is_tensor(im): return im.to(DEVICE)
    from transformers import BatchFeature
    return BatchFeature({k: v.to(DEVICE) for k, v in im.items()})   # BatchFeature.to() moves itself in place
def fwd(model, source, im):
    return model(im) if source == "timm" else model(**im).logits
def clone_im(im):
    if torch.is_tensor(im): return im.clone()
    from transformers import BatchFeature
    return BatchFeature({k: v.clone() for k, v in im.items()})
def _bsz(args, kw):
    if args and torch.is_tensor(args[0]): return int(args[0].shape[0])
    return int(kw["pixel_values"].shape[0])
def _hidden(args, kw):
    if args and torch.is_tensor(args[0]): return args[0]
    if "hidden_states" in kw: return kw["hidden_states"]
    raise RuntimeError("block called without a positional tensor or hidden_states")
def _with_hidden(args, kw, y):
    if args and torch.is_tensor(args[0]): return (y,) + tuple(args[1:]), kw
    kw = dict(kw); kw["hidden_states"] = y; return args, kw
def img_hashes(cache_or_dl, out):
    for im, _ in cache_or_dl:
        t = pix(im)
        for i in range(t.shape[0]): out.append(hashlib.sha1(t[i].contiguous().numpy().tobytes()).hexdigest()[:16])
        yield im, _

In [ ]:
# rotary helpers: from W5 / RB-0 / RB-5
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

class RopeGlobal:          # W8 / RB-0's floor: rotations off in every block
    def __init__(s, model, tf): s.name, s.mod = find_rope(model); s.tf = tf; s.handles = []
    def __enter__(s):
        if s.mod is None: raise RuntimeError("no get_embed module")
        s.orig = s.mod.get_embed; o, t = s.orig, s.tf
        s.mod.get_embed = (lambda _o=o, _t=t: (lambda *a, **k: _t(_o(*a, **k))))()
        s.handles = [1]; return s
    def __exit__(s, *e): s.mod.get_embed = s.orig; s.handles = []

class RopeComposite:       # rotations off in a set of blocks (RB-0 / RB-5)
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class APEZero:             # W1b / W8 / RB-0's APE floor: the whole learned embedding set to zero
    def __init__(s, model): s.model = model; s.handles = []
    def __enter__(s):
        cands = [n for n, _ in s.model.named_parameters() if n.endswith("pos_embed") or n.endswith("position_embeddings")]
        if not cands: raise RuntimeError("no positional-embedding parameter")
        s.name = sorted(cands, key=len)[0]; parts = s.name.split("."); m = s.model
        for p in parts[:-1]: m = getattr(m, p)
        s.parent, s.attr = m, parts[-1]
        pe = getattr(s.parent, s.attr); s.saved = pe.detach().clone()
        with torch.no_grad(): pe.zero_()
        s.handles = [1]; return s
    def __exit__(s, *e):
        with torch.no_grad(): getattr(s.parent, s.attr).copy_(s.saved)
        s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

class Multi:               # several interventions at once; hooks are counted across all of them
    def __init__(s, *ctxs): s.ctxs = ctxs; s.handles = []
    def __enter__(s):
        for c in s.ctxs: c.__enter__()
        s.handles = [h for c in s.ctxs for h in (getattr(c, "handles", []) or [])]; return s
    def __exit__(s, *e):
        for c in reversed(s.ctxs): c.__exit__(*e)
        s.handles = []

## The edits
`derive_params` and `apply_edit` are RB-8's, unchanged (RB-9 uses `mean` and its control `offset`). `ResidualEdit` / `AllEdit` are
RB-8's block-input edit (used here to rebuild RB-8's arm). New: `MlpEdit` (the stream between attention and MLP), `OutEdit` (the last
block's output) and `AllSites`. Every class is gated on synthetic data and on the three real block classes below, with the same code
the model run uses.

In [ ]:
def derangement(P, seed=SCRAMBLE_SEED):
    rng = np.random.default_rng(seed)
    while True:
        p = rng.permutation(P)
        if not np.any(p == np.arange(P)): return p

def derive_params(mu, Vp, site):
    """mu: [P, D] per-slot mean (float64), Vp: [P] per-slot E||h - mu_p||^2. Returns float64 CPU tensors."""
    mu = torch.as_tensor(mu, dtype=torch.float64); Vp = torch.as_tensor(Vp, dtype=torch.float64)
    P, D = mu.shape
    mubar = mu.mean(0); c = mu - mubar
    rho = Vp.clamp_min(1e-12).sqrt(); rhobar = Vp.mean().clamp_min(1e-12).sqrt()
    _, sv, Vh = torch.linalg.svd(c, full_matrices=False)
    e = (sv ** 2).cumsum(0) / (sv ** 2).sum().clamp_min(1e-30)
    r = int(min(int((e < SUBSPACE_ENERGY).sum().item()) + 1, len(sv)))
    V = Vh[:r].T.contiguous()
    g = torch.Generator().manual_seed(SUBSPACE_SEED + site)
    Q, _ = torch.linalg.qr(torch.randn(D, r, generator=g, dtype=torch.float64))
    perm = torch.as_tensor(derangement(P), dtype=torch.long)
    return {"mu": mu, "mubar": mubar, "c": c, "ratio": rhobar / rho, "V": V, "Q": Q.contiguous(), "perm": perm, "r": r,
            "energy_kept_by_V": float(e[r - 1])}

def params_to(p, device, dtype=torch.float32):
    return {k: (v.to(device=device, dtype=dtype) if torch.is_tensor(v) and v.is_floating_point() else
                (v.to(device) if torch.is_tensor(v) else v)) for k, v in p.items()}

def apply_edit(h, kind, p):
    """h: [B, P, D] patch tokens of the residual stream entering the edited block."""
    if kind == "mean":     return h - p["c"]
    if kind == "scale":    return p["mubar"] + (h - p["mu"]) * p["ratio"][:, None]
    if kind == "sub":      return h - ((h - p["mubar"]) @ p["V"]) @ p["V"].T
    if kind == "randsub":  return h - ((h - p["mubar"]) @ p["Q"]) @ p["Q"].T
    if kind == "scramble": return h - p["c"] + p["c"][p["perm"]]
    if kind == "offset":   return h - p["c"][p["perm"]]
    raise ValueError(kind)

class ResidualEdit:
    """Pre-hook on block s: rewrite the patch tokens entering it (prefix tokens untouched). Readout s sees the edited
    state."""
    def __init__(s, model, source, block, kind, prm, n_prefix):
        s.blk = get_vit_blocks(model, source)[block]; s.kind, s.prm, s.npf = kind, prm, n_prefix; s.handles = []
    def __enter__(s):
        def pre(mod, args, kw):
            x = _hidden(args, kw)
            y = x.clone(); y[:, s.npf:, :] = apply_edit(x[:, s.npf:, :], s.kind, s.prm)
            return _with_hidden(args, kw, y)
        s.handles = [s.blk.register_forward_pre_hook(pre, with_kwargs=True)]; return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

# Part A statistics from the fitting sums (also gated on synthetic data)
def stats_from_sums(S, Se, So, Q2, n, ne, no, H, W):
    """S, Se, So: [P, D] sums (all / even / odd images), Q2: [P] sum of squared norms. float64 numpy."""
    mu = S / n; mubar = mu.mean(0); c = mu - mubar; P = mu.shape[0]
    Vp = Q2 / n - (mu ** 2).sum(1)
    tot = float((Q2 / n).mean() - (mubar ** 2).sum())                      # E ||h - mubar||^2 over tokens
    ec = float((c ** 2).sum(1).mean())
    pvf_raw = ec / tot; pvf_bc = (ec - (1 - 1 / P) * float(Vp.mean()) / n) / tot
    g = c.reshape(H, W, -1); rr = g.mean(1, keepdims=True); cc = g.mean(0, keepdims=True); it = g - rr - cc
    e_r, e_c, e_i = float((rr ** 2).sum() * W), float((cc ** 2).sum() * H), float((it ** 2).sum())
    es = max(e_r + e_c + e_i, 1e-30)
    sv = np.linalg.svd(c, compute_uv=False); cum = np.cumsum(sv ** 2) / max((sv ** 2).sum(), 1e-30)
    r99 = int(min(int((cum < SUBSPACE_ENERGY).sum()) + 1, len(sv)))
    ce = Se / ne - (Se / ne).mean(0); co = So / no - (So / no).mean(0)
    split = float((ce * co).sum() / max(np.sqrt((ce ** 2).sum() * (co ** 2).sum()), 1e-30))
    return {"pvf_raw": pvf_raw, "pvf_bc": pvf_bc, "row_frac": e_r / es, "col_frac": e_c / es, "int_frac": e_i / es,
            "r99": r99, "split_half_cos": split, "total_var": tot, "n": int(n)}, mu, Vp, c

class AllEdit:
    """ResidualEdit at several blocks at once; rotations untouched."""
    def __init__(s, model, source, blocks, kind, prms, n_prefix):
        s.edits = [ResidualEdit(model, source, b, kind, prms[b], n_prefix) for b in blocks]; s.handles = []
    def __enter__(s):
        for e in s.edits: e.__enter__()
        s.handles = [h for e in s.edits for h in e.handles]; return s
    def __exit__(s, *e):
        for x in reversed(s.edits): x.__exit__(*e)
        s.handles = []

# RB-9: the stream between attention and MLP, and the last block's output
ALLOWED_BLOCKS = {"EvaBlock", "Block", "ViTLayer"}   # pre-norm blocks: out = r + MLP(norm2(r)), r = x + Attn(norm1(x))

def mlp_norm(blk, source):
    """The norm that feeds the block's MLP; its input is the residual stream after attention."""
    return blk.layernorm_after if source == "transformers" else blk.norm2

def _edit_out(out, kind, prm, npf):
    o = out[0] if isinstance(out, tuple) else out
    y = o.clone(); y[:, npf:, :] = apply_edit(o[:, npf:, :], kind, prm)
    return (y,) + tuple(out[1:]) if isinstance(out, tuple) else y

class MlpEdit:
    """Edit the stream after block b's attention, before its MLP: a pre-hook on the MLP's norm and a post-hook on the
    block give out = r' + MLP(norm2(r')). Exact for a per-slot constant change."""
    def __init__(s, model, source, block, kind, prm, n_prefix):
        s.blk = get_vit_blocks(model, source)[block]; s.nrm = mlp_norm(s.blk, source)
        s.kind, s.prm, s.npf = kind, prm, n_prefix; s.handles = []
    def __enter__(s):
        def pre(mod, args):
            x = args[0]; y = x.clone(); y[:, s.npf:, :] = apply_edit(x[:, s.npf:, :], s.kind, s.prm)
            return (y,) + tuple(args[1:])
        def post(mod, inp, out): return _edit_out(out, s.kind, s.prm, s.npf)
        s.handles = [s.nrm.register_forward_pre_hook(pre), s.blk.register_forward_hook(post)]; return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class OutEdit:
    """Edit the last block's output (the stream the final norm reads)."""
    def __init__(s, model, source, block, kind, prm, n_prefix):
        s.blk = get_vit_blocks(model, source)[block]; s.kind, s.prm, s.npf = kind, prm, n_prefix; s.handles = []
    def __enter__(s):
        s.handles = [s.blk.register_forward_hook(lambda mod, inp, out: _edit_out(out, s.kind, s.prm, s.npf))]; return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

SITE_CLS = {"attn": ResidualEdit, "mlp": MlpEdit, "out": OutEdit}
def hooks_of(site): return 2 if site[0] == "mlp" else 1

class AllSites:
    """Edits at a list of sites at once, registered in forward order (so on the last block the MLP edit runs before `out`)."""
    def __init__(s, model, source, sites, kind, prms, n_prefix):
        s.edits = [SITE_CLS[x[0]](model, source, x[1], kind, prms[x], n_prefix) for x in sites]; s.handles = []
    def __enter__(s):
        for e in s.edits: e.__enter__()
        s.handles = [h for e in s.edits for h in e.handles]; return s
    def __exit__(s, *e):
        for x in reversed(s.edits): x.__exit__(*e)
        s.handles = []

def mlp_branch(blk, source, r):
    """Reference for the structure gate: what the block adds to the stream r through its MLP (eval mode)."""
    name = type(blk).__name__
    if name == "EvaBlock":
        m = blk.mlp(blk.norm2(r))
        if blk.gamma_2 is not None: m = blk.gamma_2 * m
        return blk.drop_path2(m)
    if name == "Block":   return blk.drop_path2(blk.ls2(blk.mlp(blk.norm2(r))))
    if name == "ViTLayer": return blk.dropout(blk.mlp(blk.layernorm_after(r)))
    raise ValueError(name)


In [ ]:
# algebra gates on synthetic data (same functions the model run uses)
_g = torch.Generator().manual_seed(0)
_P, _D, _N = 16, 32, 4000
_ctrue = torch.randn(_P, _D, generator=_g, dtype=torch.float64) * 2.0
_sc = 0.5 + torch.rand(_P, generator=_g, dtype=torch.float64) * 2.0
_X = torch.randn(_N, _P, _D, generator=_g, dtype=torch.float64) * _sc[:, None] + _ctrue + 3.0
_mu = _X.mean(0); _Vp = ((_X - _mu) ** 2).sum(-1).mean(0)
_pr = derive_params(_mu, _Vp, site=4)
_slotmean = lambda Y: Y.mean(0)
_m = apply_edit(_X, "mean", _pr)
gate("mean erasure: every slot's mean equals the average over slots", "edit-algebra",
     torch.allclose(_slotmean(_m), _pr["mubar"].expand(_P, _D), atol=1e-9))
_s = apply_edit(_X, "scale", _pr); _sv = ((_s - _pr["mubar"]) ** 2).sum(-1).mean(0)
gate("mean+scale erasure: equal slot means and equal slot spread", "edit-algebra",
     torch.allclose(_slotmean(_s), _pr["mubar"].expand(_P, _D), atol=1e-9) and torch.allclose(_sv, _sv.mean().expand(_P), rtol=1e-9),
     f"spread range {float(_sv.min()):.4f}-{float(_sv.max()):.4f}")
_u = apply_edit(_X, "sub", _pr); _cu = _slotmean(_u) - _slotmean(_u).mean(0)
gate("subspace erasure removes >= 99% of the positional component's energy", "edit-algebra",
     float((_cu ** 2).sum() / (_pr["c"] ** 2).sum()) <= 1 - SUBSPACE_ENERGY + 1e-9,
     f"r={_pr['r']} of {_D}; energy left {float((_cu ** 2).sum() / (_pr['c'] ** 2).sum()):.2e}")
_Q, _V = _pr["Q"], _pr["V"]
gate("random subspace: orthonormal, same dimension as the positional subspace, reproducible", "edit-algebra",
     _Q.shape == _V.shape and torch.allclose(_Q.T @ _Q, torch.eye(_pr["r"], dtype=torch.float64), atol=1e-10)
     and torch.equal(_Q, derive_params(_mu, _Vp, site=4)["Q"]) and not torch.equal(_Q, derive_params(_mu, _Vp, site=2)["Q"]))
_rr = apply_edit(_X, "randsub", _pr); _cr = _slotmean(_rr) - _slotmean(_rr).mean(0)
gate("random-subspace control leaves most of the positional component (it is a control, not an erasure)", "edit-algebra",
     float((_cr ** 2).sum() / (_pr["c"] ** 2).sum()) > 0.3, f"energy left {float((_cr ** 2).sum() / (_pr['c'] ** 2).sum()):.3f}")
_q = apply_edit(_X, "scramble", _pr); _pm = _pr["perm"]
gate("scramble: slot p carries slot pi(p)'s component; pi has no fixed point", "edit-algebra",
     torch.allclose(_slotmean(_q), _pr["mubar"] + _pr["c"][_pm], atol=1e-9) and not bool((_pm == torch.arange(_P)).any())
     and sorted(_pm.tolist()) == list(range(_P)))
gate("derangement is reproducible", "edit-algebra", np.array_equal(derangement(196), derangement(196)))

class _StubM(nn.Module):
    def __init__(s): super().__init__(); s.blocks = nn.ModuleList([nn.Identity() for _ in range(3)])
_sm = _StubM(); _x = torch.randn(2, 3 + _P, _D, dtype=torch.float64)
with ResidualEdit(_sm, "timm", 1, "mean", _pr, 3) as _re:
    _y = _sm.blocks[1](_x); _y0 = _sm.blocks[0](_x); _nh = len(_re.handles)
gate("edit hook: prefix tokens untouched, patch tokens edited, other blocks untouched, one hook", "edit-algebra",
     torch.equal(_y[:, :3], _x[:, :3]) and torch.allclose(_y[:, 3:], _x[:, 3:] - _pr["c"]) and torch.equal(_y0, _x) and _nh == 1)
gate("edit hook removed on exit", "edit-algebra", torch.equal(_sm.blocks[1](_x), _x))

# Part A statistics: a pure row component, a pure column component, and no component at all
_H = _W = 4; _n = 6000
def _sums(X):
    ev = np.arange(X.shape[0]) % 2 == 0
    return X.sum(0), X[ev].sum(0), X[~ev].sum(0), (X ** 2).sum(-1).sum(0), X.shape[0], int(ev.sum()), int((~ev).sum())
_rng = np.random.default_rng(1)
_row = np.repeat(_rng.normal(size=(_H, 1, 8)), _W, 1).reshape(16, 8) * 1.5
_col = np.repeat(_rng.normal(size=(1, _W, 8)), _H, 0).reshape(16, 8) * 1.5
_A0 = stats_from_sums(*_sums(_rng.normal(size=(_n, 16, 8))), _H, _W)[0]
_Ar = stats_from_sums(*_sums(_rng.normal(size=(_n, 16, 8)) + _row), _H, _W)[0]
_Ac = stats_from_sums(*_sums(_rng.normal(size=(_n, 16, 8)) + _col), _H, _W)[0]
_true_r = float(((_row - _row.mean(0)) ** 2).sum(1).mean()); _true_r /= (_true_r + 8.0)
gate("Part A: no positional component gives bias-corrected PVF ~ 0", "partA-algebra", abs(_A0["pvf_bc"]) < 0.002,
     f"raw {_A0['pvf_raw']:.4f}, corrected {_A0['pvf_bc']:+.5f}")
gate("Part A: bias-corrected PVF recovers the true value; row / column split is right", "partA-algebra",
     abs(_Ar["pvf_bc"] - _true_r) < 0.01 and _Ar["row_frac"] > 0.95 and _Ac["col_frac"] > 0.95 and _Ar["split_half_cos"] > 0.9,
     f"PVF {_Ar['pvf_bc']:.4f} vs {_true_r:.4f}; row {_Ar['row_frac']:.3f}, col {_Ac['col_frac']:.3f}")

_x = torch.randn(2, 3 + _P, _D, dtype=torch.float64)
with AllEdit(_sm, "timm", [0, 1], "mean", {0: _pr, 1: _pr}, 3) as _ae:
    _a0, _a1, _a2 = _sm.blocks[0](_x), _sm.blocks[1](_x), _sm.blocks[2](_x); _na = len(_ae.handles)
gate("AllEdit: the listed blocks are edited, the others are not, one hook per block", "edit-algebra",
     torch.allclose(_a0[:, 3:], _x[:, 3:] - _pr["c"]) and torch.allclose(_a1[:, 3:], _x[:, 3:] - _pr["c"]) and torch.equal(_a2, _x)
     and torch.equal(_a0[:, :3], _x[:, :3]) and _na == 2)
gate("AllEdit hooks removed on exit", "edit-algebra", torch.equal(_sm.blocks[0](_x), _x) and torch.equal(_sm.blocks[1](_x), _x))
_o = apply_edit(_X, "offset", _pr)
gate("offset control: subtracts another slot's component, same vector sizes as the mean edit, position left in place", "edit-algebra",
     torch.allclose(_slotmean(_o), _pr["mu"] - _pr["c"][_pm], atol=1e-9)
     and torch.allclose((_X - _o)[0].norm(dim=-1).sort()[0], _pr["c"].norm(dim=-1).sort()[0], atol=1e-9)
     and float(((_slotmean(_o) - _slotmean(_o).mean(0)) ** 2).sum()) > 0.5 * float((_pr["c"] ** 2).sum()))

# RB-9 site edits on real block classes (tiny, random weights, CPU, float64)
from timm.models.eva import EvaBlock as _EvaBlock
from timm.models.vision_transformer import Block as _VitBlock
from transformers import ViTConfig as _VC
from transformers.models.vit.modeling_vit import ViTLayer as _ViTLayer
def _stub_blocks(blks, source):
    m = nn.Module()
    if source == "timm": m.blocks = nn.ModuleList(blks)
    else: m.layers = nn.ModuleList(blks)
    return m
torch.manual_seed(3)
_Dm, _Pm, _npf = 32, 16, 1
_cands = [("EvaBlock", "timm", lambda: _EvaBlock(_Dm, 4, init_values=0.1)),
          ("Block", "timm", lambda: _VitBlock(_Dm, 4, init_values=0.1)),
          ("ViTLayer", "transformers", lambda: _ViTLayer(_VC(hidden_size=_Dm, num_attention_heads=4, intermediate_size=64)))]
_gp = derive_params(torch.randn(_Pm, _Dm, dtype=torch.float64), torch.rand(_Pm, dtype=torch.float64) + 0.5, site=4)
_gp0 = dict(_gp); _gp0["c"] = torch.zeros_like(_gp["c"])
for _nm, _src, _mk in _cands:
    _blk = _mk().double().eval(); _sm = _stub_blocks([_blk], _src)
    _x = torch.randn(3, _npf + _Pm, _Dm, dtype=torch.float64)
    with torch.no_grad():
        _plain = _blk(_x); _plain = _plain[0] if isinstance(_plain, tuple) else _plain
        _cap = {}
        _h = mlp_norm(_blk, _src).register_forward_pre_hook(lambda m, a: _cap.__setitem__("r", a[0].clone()))
        _blk(_x); _h.remove()
        for _kind in ("mean", "offset"):
            with MlpEdit(_sm, _src, 0, _kind, _gp, _npf) as _me:
                _y = _blk(_x); _y = _y[0] if isinstance(_y, tuple) else _y; _nh = len(_me.handles)
            _r = _cap["r"].clone(); _r[:, _npf:] = apply_edit(_cap["r"][:, _npf:], _kind, _gp)
            _ref = _r + mlp_branch(_blk, _src, _r)
            _d = float((_y - _ref).abs().max() / _ref.abs().max())
            gate(f"MLP-site edit [{_kind}] on a {_nm}: output = edited stream + MLP(norm(edited stream)), prefix untouched, 2 hooks", "edit-algebra",
                 _d < 1e-12 and torch.equal(_y[:, :_npf], _plain[:, :_npf]) and _nh == 2, f"rel. diff {_d:.1e}")
        with MlpEdit(_sm, _src, 0, "mean", _gp0, _npf):
            _y0 = _blk(_x); _y0 = _y0[0] if isinstance(_y0, tuple) else _y0
        gate(f"MLP-site edit with a zero component leaves a {_nm} bit-identical", "edit-algebra", torch.equal(_y0, _plain))
        _y1 = _blk(_x); _y1 = _y1[0] if isinstance(_y1, tuple) else _y1
        gate(f"MLP-site hooks removed on exit ({_nm})", "edit-algebra", torch.equal(_y1, _plain))
# last block: MLP edit, then `out` (three stub blocks, sites in forward order)
_blks = [_EvaBlock(_Dm, 4, init_values=0.1).double().eval() for _ in range(3)]; _sm = _stub_blocks(_blks, "timm")
_sites = [("mlp", 0), ("attn", 1), ("mlp", 1), ("attn", 2), ("mlp", 2), ("out", 2)]
_prm = {s_: derive_params(torch.randn(_Pm, _Dm, dtype=torch.float64), torch.rand(_Pm, dtype=torch.float64) + 0.5, site=4) for s_ in _sites}
_x = torch.randn(2, _npf + _Pm, _Dm, dtype=torch.float64)
with torch.no_grad():
    with AllSites(_sm, "timm", _sites, "mean", _prm, _npf) as _as:
        _h = _x
        for _b in _blks: _h = _b(_h)
        _na = len(_as.handles)
    _ref = _x
    for _i, _b in enumerate(_blks):
        if ("attn", _i) in _prm: _ref = _ref.clone(); _ref[:, _npf:] = _ref[:, _npf:] - _prm[("attn", _i)]["c"]
        _cap = {}
        _hh = _b.norm2.register_forward_pre_hook(lambda m, a: _cap.__setitem__("r", a[0].clone())); _b(_ref); _hh.remove()
        _r = _cap["r"].clone(); _r[:, _npf:] = _r[:, _npf:] - _prm[("mlp", _i)]["c"]
        _ref = _r + mlp_branch(_b, "timm", _r)
    _ref[:, _npf:] = _ref[:, _npf:] - _prm[("out", 2)]["c"]
gate("AllSites: attention, MLP and output sites applied in forward order equal a step-by-step reference; hook count", "edit-algebra",
     float((_h - _ref).abs().max() / _ref.abs().max()) < 1e-12 and _na == sum(hooks_of(s_) for s_ in _sites),
     f"rel. diff {float((_h - _ref).abs().max() / _ref.abs().max()):.1e}; {_na} hooks")
gate("site lists: RoPE 24 sites starting at mlp0, hybrid / APE 25 starting at attn0, both ending with out", "edit-algebra",
     len(sites_of("naver")) == 24 and sites_of("naver")[0] == ("mlp", 0) and len(sites_of("deit3")) == 25
     and sites_of("deit3")[0] == ("attn", 0) and sites_of("naver")[-1] == ("out", 11) and sites_of("rope_ape")[:2] == [("attn", 0), ("mlp", 0)])


In [ ]:
# image cache: decode once per model, replay through a real DataLoader (RB-0, source-aware)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(clone_im(im), torch.as_tensor(lb).clone()) for im, lb in dl]
    got = sum(int(lb.numel()) for _, lb in out)
    print(f"    cached {got} images in {time.time() - t:.0f}s")
    return out

def cache_hashes(cache):
    h = []; [None for _ in img_hashes(cache, h)]; return h

In [ ]:
# measurement: RB-8's measure(), plus a readout at every block's MLP input
def final_norm_module(model, source):
    if source == "transformers": return model.vit.layernorm, "vit.layernorm"
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

N_READOUTS = MLP0 + N_BLOCKS                    # 0-11 block inputs (after norm1), 12 FINAL, 13-24 MLP inputs (after norm2)
def readout_label(r): return f"b{r}" if r < FINAL else ("final" if r == FINAL else f"mlp{r - MLP0}")

def measure(M, ctx, capture=()):
    """One RPI sweep over the cached repack images under `ctx`: SSDC at every readout, plus captured patch tokens.
    Readout hooks are registered after ctx's, so they see the edited stream."""
    model, source, n_prefix = M["model"], M["source"], M["n_prefix"]
    cache = M["cache"]["ssdc"]
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    cap = {r: [] for r in capture}
    def keep(r, tok):
        if r in cap: cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())
    t0 = time.time(); handles = []
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            handles += _register_ssdc_accumulator_hooks(model, source, acc)        # the repo's own readout, keys 0-11
            handles.append(model.register_forward_pre_hook(
                lambda m, args, kw: st.update(cur=st["next"], next=st["next"] + _bsz(args, kw)), with_kwargs=True))
            for i in range(N_BLOCKS):
                if i in cap:
                    handles.append(get_block_attention(blocks[i], source).register_forward_hook(
                        lambda m, inp, out, i=i: keep(i, inp[0].detach())))
                def mlp_hook(m, inp, out, i=i):
                    o = out.detach(); acc.add(MLP0 + i, o); keep(MLP0 + i, o)
                handles.append(mlp_norm(blocks[i], source).register_forward_hook(mlp_hook))
            def final_hook(m, inp, out):
                o = out[0] if isinstance(out, tuple) else out
                t = fnorm(o.detach()); acc.add(FINAL, t); keep(FINAL, t)
            handles.append(blocks[-1].register_forward_hook(final_hook))
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(N_READOUTS)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"]),
           "runtime_s": round(time.time() - t0, 1)}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

# probes: W5's per-block-seeded protocol from RB-0 / RB-8 (seed TORCH_SEED + readout)
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        peak = max(peak, correct / Xte.shape[0])
    del head, opt
    return float(peak)

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block):
    """Exact / row / column probes, trained on 80% of the evaluation images and scored on the other 20%."""
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, _ = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name, (lab, nc) in labels.items():
        y = lab.repeat(n_images)
        out[name] = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

def measure_with_probes(M, mk_ctx, readouts):
    """SSDC from the first sweep; probes in chunks of MAX_CAPTURE readouts, one sweep each (bounds CPU RAM)."""
    H, W = M["grid"]; ro = sorted(readouts); chunks = [ro[i:i + MAX_CAPTURE] for i in range(0, len(ro), MAX_CAPTURE)] or [[]]
    first, probes, same = None, {}, True
    for ch in chunks:
        r = measure(M, mk_ctx(), capture=tuple(ch))
        if first is None: first = r
        else: same = same and r["ssdc"] == first["ssdc"] and r["hooks"] == first["hooks"]
        for b in ch: probes[str(b)] = probe_block(r["acts"][b], H, W, b)
        r.pop("acts", None); del r
    first["probes"] = probes; first["sweeps"] = len(chunks); first["sweeps_identical"] = bool(same)
    return first


In [ ]:
# top-1 on the cached ILSVRC images: RB-0's / RB-8's accuracy(), source-aware
def accuracy(M, ctx):
    model, source, cache = M["model"], M["source"], M["cache"]["acc"]
    model.eval(); correct = []
    torch.manual_seed(TORCH_SEED)
    with ctx:
        hooks = len(getattr(ctx, "handles", []) or [])
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                pred = fwd(model, source, to_dev(images)).argmax(-1).cpu()
                correct.append(pred == torch.as_tensor(labels).cpu())
    c = torch.cat(correct)
    return float(c.float().mean()), c.numpy().astype(np.uint8), hooks

# the fit: per-slot statistics of the stream at one site, under RPI
def fit_cache(M, tag, n):
    t0 = time.time()
    dl = prep_data(FIT_SRC[tag][0](), M["processor"], M["source"], corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(clone_im(im), torch.as_tensor(lb).clone()) for im, lb in dl]
    print(f"    fit images [{tag}, {n}] cached in {time.time() - t0:.0f}s", flush=True)
    return out

class _StopForward(Exception):
    pass

def _rpi_sweep(model, source, cache, stop_at_site):
    """predict(RPI=True)'s loop, rewritten so the forward can stop once the site is read: same DataLoader, same model
    call, one RPI permutation per batch from the global generator."""
    def RPI_hook(module, input, output):
        B, C, H, W = output.shape
        out = output.view(B, C, -1).contiguous()
        perm = torch.randperm(out.shape[-1])
        return out[:, :, perm].view(B, C, H, W)
    h = get_patch_embed_conv(model, source).register_forward_hook(RPI_hook)
    model.eval()
    try:
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                images = to_dev(images)
                try:
                    if source == "transformers": model(**images)
                    else: model(images)
                except _StopForward:
                    pass
    finally:
        h.remove()

def site_stats(M, fcache, site, ctx, early_stop=True):
    """Per-slot mean and spread of the stream at `site` under `ctx`. The reading hook runs after ctx's hooks, so it
    sees every edit in ctx."""
    model, source, npf = M["model"], M["source"], M["n_prefix"]
    blk = get_vit_blocks(model, source)[site[1]]; a = {}
    def add(x):
        x = x[:, npf:, :].detach().double()
        if not a:
            z = lambda *s_: torch.zeros(*s_, dtype=torch.float64, device=x.device)
            a.update(S=z(*x.shape[1:]), Se=z(*x.shape[1:]), So=z(*x.shape[1:]), Q2=z(x.shape[1]), n=0)
        ev = (torch.arange(a["n"], a["n"] + x.shape[0], device=x.device) % 2) == 0
        a["S"] += x.sum(0); a["Se"] += x[ev].sum(0); a["So"] += x[~ev].sum(0); a["Q2"] += (x * x).sum(-1).sum(0); a["n"] += x.shape[0]
        if early_stop: raise _StopForward()
    torch.manual_seed(TORCH_SEED + 1); np.random.seed((TORCH_SEED + 1) % (2 ** 31))
    with ctx:
        if site[0] == "attn":
            h = blk.register_forward_pre_hook(lambda m, args, kw: add(_hidden(args, kw)), with_kwargs=True)
        elif site[0] == "mlp":
            h = mlp_norm(blk, source).register_forward_pre_hook(lambda m, args: add(args[0]))
        else:
            h = blk.register_forward_hook(lambda m, inp, out: add(out[0] if isinstance(out, tuple) else out))
        try:
            if early_stop: _rpi_sweep(model, source, fcache, site)
            else: predict(model, cached_loader(fcache), source, RPI=True, half=False)
        finally:
            h.remove()
    n = a["n"]; S, Q2 = a["S"].cpu().numpy(), a["Q2"].cpu().numpy()
    mu = S / n; Vp = Q2 / n - (mu ** 2).sum(1)
    ne, no = (n + 1) // 2, n // 2
    ce = a["Se"].cpu().numpy() / ne; co = a["So"].cpu().numpy() / no; ce = ce - ce.mean(0); co = co - co.mean(0)
    cos = float((ce * co).sum() / max(np.sqrt((ce ** 2).sum() * (co ** 2).sum()), 1e-30))
    return mu, Vp, n, cos


In [ ]:
# per-model setup
def _smooth_pe(pe, G=14, scale=3.0):
    """Test mode only: a smooth 2-D sin-cos table on the patch slots, so a tiny random APE model has spatial structure."""
    D = pe.shape[-1]; q = D // 4; om = 1.0 / (100.0 ** (torch.arange(q, dtype=torch.float64) / q))
    ii, jj = torch.meshgrid(torch.arange(G, dtype=torch.float64), torch.arange(G, dtype=torch.float64), indexing="ij")
    a, b = ii.reshape(-1, 1) * om, jj.reshape(-1, 1) * om
    t = torch.cat([a.sin(), a.cos(), b.sin(), b.cos()], 1)
    pe.zero_(); pe[0, -G * G:, :t.shape[1]] = (t * scale).to(pe.dtype)

TEST_INIT_SEED = {"naver": 101, "DINOv3": 102, "small": 103, "reg1_gap": 104}   # rope_ape: 0, as in the test W8
def load_model(key):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        ncls = 10 if spec["classifier"] else 0
        if spec["loader"] == "hf":
            from transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor
            torch.manual_seed(201)
            cfg = ViTConfig(hidden_size=96, num_hidden_layers=12, num_attention_heads=6, intermediate_size=192,
                            image_size=INPUT_SIZE, patch_size=16, num_labels=10)
            model = ViTForImageClassification(cfg).to(DEVICE).eval()
            with torch.no_grad(): _smooth_pe(model.vit.embeddings.position_embeddings)
            return model, ViTImageProcessor(size={"height": INPUT_SIZE, "width": INPUT_SIZE}), "transformers"
        tiny = dict(embed_dim=96, num_heads=6 if key != "naver" else 12, pretrained=False, dynamic_img_size=True, num_classes=ncls)
        if key == "naver": tiny["embed_dim"] = 192
        torch.manual_seed(TEST_INIT_SEED.get(key, 0) if spec["family"] != "ape" else 200 + list(MODEL_SPECS).index(key))
        model = timm.create_model(spec["name"].split(".")[0], **tiny).to(DEVICE).eval()
        with torch.no_grad():
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                if spec["family"] == "ape":
                    for ln in ("ls1", "ls2"):
                        g_ = getattr(getattr(blk, ln, None), "gamma", None)
                        if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
                for pn in ("q_proj", "k_proj"):
                    m_ = getattr(blk.attn, pn, None)
                    if m_ is not None: m_.weight.mul_(4.0)
            if spec["family"] == "ape": _smooth_pe(model.pos_embed)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, INPUT_SIZE, INPUT_SIZE)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    if spec["loader"] == "rope": return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=INPUT_SIZE)
    if spec["loader"] == "hf":   return load_ape(spec["name"], device=DEVICE, half=False)
    return load_ape_timm(spec["name"], device=DEVICE, half=False)

def first_layer_rel_diff(M):
    model, source = M["model"], M["source"]
    conv = get_patch_embed_conv(model, source); grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    try:
        with torch.no_grad(): fwd(model, source, to_dev(M["cache"]["ssdc"][0][0]))
    finally:
        h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    n = int(grab["x"].shape[0]); del grab, y64
    return v, n

def setup_model(key):
    spec = MODEL_SPECS[key]
    print("\n" + "=" * 78 + f"\n{key}  ({spec['name']})\n" + "=" * 78, flush=True)
    model, processor, source = load_model(key)
    model.eval().float()
    n_prefix = 1 if source == "transformers" else int(getattr(model, "num_prefix_tokens", 1))
    if has_rope(key):
        rh, _, nh = attn_attrs(model, source)
    else:
        nh = model.config.num_attention_heads if source == "transformers" else model.blocks[0].attn.num_heads
    M = {"key": key, "model": model, "source": source, "processor": processor, "n_prefix": n_prefix,
         "n_heads": int(nh), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, N_SSDC)
    if spec["classifier"]:
        M["cache"]["acc"] = build_cache(DS_ACC, processor, source, N_ACC)
    M["eval_hashes"] = {"rep": cache_hashes(M["cache"]["ssdc"])}
    if spec["classifier"]: M["eval_hashes"]["acc"] = cache_hashes(M["cache"]["acc"])
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(pix(M["cache"]["ssdc"][0][0])[:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg)
    gate("forward grid is 14x14", key, (Hg, Wg) == (14, 14), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model, source); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    print(f"    source {source} | prefix tokens {n_prefix} | heads {nh} | final norm {fn_name}")
    ra = (RB0.get(key) or {}).get("A")
    if ra:
        gate("num_heads matches RB-0", key, M["n_heads"] == ra["n_heads"], f"{M['n_heads']} vs {ra['n_heads']}")
    if has_rope(key):
        _, rmod = find_rope(model)
        emb0 = rmod.get_embed(shape=(Hg, Wg))
        gate("rope rows equal patch tokens (196) and the table has no head dimension", key,
             int(emb0.shape[0]) == 196 and emb0.dim() == 2, f"shape {tuple(emb0.shape)}")
    else:
        try:
            with APEZero(model) as _z: nm = _z.name
            gate("APE floor finds the learned position embedding", key, True, nm)
        except Exception as ex:
            gate("APE floor finds the learned position embedding", key, False, repr(ex))
    if DEVICE == "cuda" and not TEST_MODE:
        v, nb = first_layer_rel_diff(M); M["first_layer_rel_diff"] = v
        if key in FIRST_LAYER_910:
            exp_ = FIRST_LAYER_910[key]
            gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value (TF32-level kernel, as in September)", key,
                 nb == BATCH_SIZE and abs(v / exp_ - 1) <= 0.01, f"{v:.4e} vs {exp_:.4e} (bit-identical: {v == exp_}; batch {nb})")
        else:
            gate(f"first-layer numerics are TF32-level, as for every model in RB-1b ({FIRST_LAYER_RANGE[0]:.0e}-{FIRST_LAYER_RANGE[1]:.0e})", key,
                 nb == BATCH_SIZE and FIRST_LAYER_RANGE[0] <= v <= FIRST_LAYER_RANGE[1], f"{v:.4e} (batch {nb})")
    else:
        gate("first-layer numerics", key, True, "test mode / CPU", skip=True)
    # block structure: the MLP-site edit assumes out = r + MLP(norm2(r)), r = stream after attention
    blocks_ = get_vit_blocks(model, source)
    names_ = sorted({type(b_).__name__ for b_ in blocks_})
    gate("every block is a pre-norm block the MLP-site edit supports (EvaBlock / Block / ViTLayer)", key,
         set(names_) <= ALLOWED_BLOCKS and len(blocks_) == N_BLOCKS, f"{names_}, {len(blocks_)} blocks")
    cnt = {"n": 0}
    hcnt = [mlp_norm(b_, source).register_forward_pre_hook(lambda m, a_: cnt.__setitem__("n", cnt["n"] + 1)) for b_ in blocks_]
    capt = {}
    hcap = [blocks_[i].register_forward_pre_hook(lambda m, args, kw, i=i: capt.__setitem__(i, (args, kw)), with_kwargs=True)
            for i in (0, 5, N_BLOCKS - 1)]
    try:
        with torch.no_grad(): fwd(model, source, to_dev(M["cache"]["ssdc"][0][0]))
    finally:
        for h_ in hcnt + hcap: h_.remove()
    nb_ = int(pix(M["cache"]["ssdc"][0][0]).shape[0])
    gate("each block's MLP norm runs exactly once per forward pass", key, cnt["n"] == N_BLOCKS, f"{cnt['n']} calls for {N_BLOCKS} blocks")
    worst, zero_ok = 0.0, True
    g_ = torch.Generator().manual_seed(11)
    for i, (args, kw) in capt.items():
        blk = blocks_[i]
        with torch.no_grad():
            x_ = _hidden(args, kw); D_ = x_.shape[-1]; P_ = x_.shape[1] - n_prefix
            c_ = (torch.randn(P_, D_, generator=g_) * float(x_[:, n_prefix:].pow(2).mean().sqrt()) * 0.05).to(x_.device)
            prm_ = {"c": c_, "perm": torch.arange(P_).roll(1).to(x_.device)}
            prm0_ = {"c": torch.zeros_like(c_), "perm": prm_["perm"]}
            stub_ = _stub_blocks([blk], source)
            plain = blk(*args, **kw); plain = plain[0] if isinstance(plain, tuple) else plain
            rr = {}
            hh = mlp_norm(blk, source).register_forward_pre_hook(lambda m, a_: rr.__setitem__("r", a_[0].clone()))
            blk(*args, **kw); hh.remove()
            with MlpEdit(stub_, source, 0, "mean", prm_, n_prefix):
                y_ = blk(*args, **kw); y_ = y_[0] if isinstance(y_, tuple) else y_
            r_ = rr["r"].clone(); r_[:, n_prefix:] = r_[:, n_prefix:] - c_
            ref_ = r_ + mlp_branch(blk, source, r_)
            worst = max(worst, float((y_ - ref_).abs().max() / ref_.abs().max()))
            with MlpEdit(stub_, source, 0, "mean", prm0_, n_prefix):
                y0_ = blk(*args, **kw); y0_ = y0_[0] if isinstance(y0_, tuple) else y0_
            zero_ok = zero_ok and torch.equal(y0_, plain)
    gate(f"on real activations (blocks 0, 5, {N_BLOCKS - 1}): the hooked block equals edited stream + MLP(norm(edited stream))", key,
         worst <= STRUCT_TOL, f"max rel. diff {worst:.1e} (float add order only; tolerance {STRUCT_TOL:.0e}); batch {nb_}")
    gate("a zero MLP-site edit leaves real blocks bit-identical", key, zero_ok)
    def streamed_ssdc(ctx):
        torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
        with ctx:
            s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=N_SSDC,
                                 batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
        return s
    for nm_, mk in [("baseline", NullCtx), ("floor", lambda: floor_ctx(M, floor_name(key)))]:
        a = streamed_ssdc(mk()); b = measure(M, mk())["ssdc"][:N_BLOCKS]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate(f"cached replay reproduces streamed evaluate_ssdc [{nm_}]", key, d <= EXACT_TOL, f"max |delta| {d:.2e}")
    if "acc" in M["cache"]:
        n_chk = min(REPLAY_CHECK_N, N_ACC)
        dl = prep_data(DS_ACC, processor, source, corruption_type=None, number_images=n_chk,
                       batch_size=BATCH_SIZE, half=False, num_workers=0)
        cs = []
        with torch.inference_mode():
            for im, lb in dl:
                cs.append(fwd(model, source, to_dev(im)).argmax(-1).cpu() == torch.as_tensor(lb))
        cs = torch.cat(cs).numpy().astype(np.uint8)
        _, cc, _ = accuracy(M, NullCtx())
        gate(f"cached replay reproduces streamed top-1 on the first {n_chk} images", key,
             np.array_equal(cs, cc[:n_chk]), f"{int((cs != cc[:n_chk]).sum())} images differ")
    return M

def floor_ctx(M, which):
    model, source = M["model"], M["source"]
    if which == "floor":      return RopeGlobal(model, IDEN) if has_rope(M["key"]) else APEZero(model)
    if which == "ape_floor":  return APEZero(model)
    if which == "both_floor": return Multi(RopeGlobal(model, IDEN), APEZero(model))
    raise ValueError(which)

In [ ]:
# bookkeeping and the fixed plan of arms
RESULTS = {}
CFG_FP = {"n_ssdc": N_SSDC, "n_acc": N_ACC, "n_fit_mean": N_FIT_MEAN, "batch": BATCH_SIZE, "seed": TORCH_SEED, "ssdc_dataset": SSDC_DATASET,
          "acc_dataset": ACC_DATASET, "metric": METRIC, "edit": "mean", "control": "offset", "fit": "sequential over sites, earlier sites edited",
          "sites": "rope: mlp0, attn/mlp 1-11, out; hybrid/ape: attn/mlp 0-11, out", "readouts": "0-11 block inputs, 12 final, 13-24 MLP inputs",
          "scramble_seed": SCRAMBLE_SEED, "probe": [PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC], "test_mode": TEST_MODE}
SESSION = {"session": RUN_STAMP, "torch": torch.__version__, "cudnn": ENV["cudnn_reported"], "python": sys.executable,
           "gpu": ENV["gpu"], "repo_commit": REPO_COMMIT}
def model_file(key): return OUT_DIR / f"rb9_{key}.json"
def img_file(key):   return OUT_DIR / f"rb9_perimage_{key}.npz"
def fit_file(key, tag): return OUT_DIR / f"rb9_fit_{key}_{tag}_mean.npz"

def load_R(key):
    f = model_file(key)
    if not (RESUME and f.exists()): return None, {}
    R = json.loads(f.read_text())
    if R.get("config") != CFG_FP:
        raise SystemExit(f"{f.name} was written with a different configuration; move it away or set RB9_RESUME=0.")
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    for n in R["acc"]:
        if n not in P: raise SystemExit(f"{img_file(key).name} lacks per-image results for {n}; set RB9_RESUME=0.")
    return R, P

def save_R(key, R, P):
    R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    if P:
        tmpi = img_file(key).with_name(img_file(key).stem + ".tmp.npz"); np.savez_compressed(tmpi, **P); os.replace(tmpi, img_file(key))
    tmp = model_file(key).with_suffix(".json.tmp"); tmp.write_text(json.dumps(R, indent=1, default=float)); os.replace(tmp, model_file(key))

def floors_of(key):
    return ["floor", "ape_floor", "both_floor"] if MODEL_SPECS[key]["family"] == "hybrid" else ["floor"]

MLP_READOUTS = tuple(MLP0 + b for b in range(N_BLOCKS))
def full_readouts(key): return tuple(sorted(site_readout(s) for s in sites_of(key)))
def control_readouts(key): return (SITE_MAIN, 11, MLP0 + SITE_MAIN, MLP0 + 11, FINAL)

def plan(key):
    """name -> (spec, probe readouts). Same names in R['ssdc'] and R['acc']. Intact probes at block inputs come from
    RB-8, so only the MLP inputs are probed here."""
    arms = {}
    def add(name, spec, ro):
        arms[("ssdc", name)] = (spec, ro)
        if MODEL_SPECS[key]["classifier"]: arms[("acc", name)] = (spec, ())
    add("baseline", ("null",), MLP_READOUTS + (FINAL,))
    for f in floors_of(key): add(f, ("floor", f), ())
    add("rb8_mean", ("rb8",), ())
    add("full_mean", ("sites", "mean"), full_readouts(key))
    add("full_offset", ("sites", "offset"), control_readouts(key))
    return arms

def expected_hooks(key, spec):
    k = spec[0]
    if k == "null": return 0
    if k == "floor": return 2 if spec[1] == "both_floor" else 1
    if k == "rb8": return len(rb8_blocks(key))
    if k == "sites": return sum(hooks_of(s) for s in sites_of(key))
    raise ValueError(spec)

def build_ctx(M, spec, kind):
    model, source = M["model"], M["source"]
    if spec[0] == "null": return NullCtx()
    if spec[0] == "floor": return floor_ctx(M, spec[1])
    tag = "rep" if kind == "ssdc" else "acc"
    if spec[0] == "rb8": return AllEdit(model, source, rb8_blocks(M["key"]), "mean", M["rb8_prm"][tag], M["n_prefix"])
    if spec[0] == "sites": return AllSites(model, source, sites_of(M["key"]), spec[1], M["prm"][tag], M["n_prefix"])
    raise ValueError(spec)


## The sequential fit
For each dataset: the sites are fitted in forward order, each with every earlier site already edited, on the 16,384 images that
follow the evaluation images (RB-8's images, checked by hash). Each sweep stops once the site has been read; the stopping sweep is gated
against RB-8's own intact fit (bit for bit) and against the repo's full predict() sweep. One fit file per dataset; every edit is rebuilt
from that file, so a resumed run uses exactly the same edit.

In [ ]:
def params_from_file(f, sites):
    z = np.load(f); return {s: params_to(derive_params(z[f"mu_{site_name(s)}"], z[f"Vp_{site_name(s)}"], site=s[1]), DEVICE) for s in sites}

def run_fit(M, R):
    key = M["key"]; model, source, npf = M["model"], M["source"], M["n_prefix"]
    tags = ["rep"] + (["acc"] if MODEL_SPECS[key]["classifier"] else [])
    R.setdefault("fit_sha256", {}); R.setdefault("fit_hashes", {}); R.setdefault("edit_params", {})
    M["prm"] = {}; sites = sites_of(key); b8 = rb8_blocks(key)
    M["rb8_prm"] = {t: {b: params_to(derive_params(RB8[key]["fit"][t][f"mu_b{b}"], RB8[key]["fit"][t][f"Vp_b{b}"], site=b), DEVICE) for b in b8}
                    for t in tags}
    for t in tags:
        f = fit_file(key, t)
        done = f.exists() and R["fit_sha256"].get(t) == sha256_file(f)
        fc = None
        if not done:
            t0 = time.time(); fc = fit_cache(M, t, N_FIT_MEAN); hs = cache_hashes(fc)
            R["fit_hashes"][f"{t}_{N_FIT_MEAN}"] = hashlib.sha1("".join(hs).encode()).hexdigest()
            gate(f"fit [{t}] has {N_FIT_MEAN} images, right after the evaluation images", key, len(hs) == N_FIT_MEAN, f"{len(hs)}")
            ov = len(set(M["eval_hashes"][t]) & set(hs)); allowed = int(np.ceil(DISJOINT_MAX_OVERLAP * N_FIT_MEAN / 2000))
            gate(f"{N_FIT_MEAN} fitting images [{t}] are disjoint from the evaluation images", key, ov <= allowed, f"{ov} shared (allowed {allowed})")
            gate(f"fitting images [{t}] are RB-8's (same image hashes)", key,
                 R["fit_hashes"][f"{t}_{N_FIT_MEAN}"] == RB8[key]["R"].get("fit_hashes", {}).get(f"{t}_{N_FIT_MEAN}"))
            # early-stop sweep vs RB-8's intact fit at its first edited block, and vs a full predict() sweep
            b0 = b8[0]
            mu, Vp, n, _ = site_stats(M, fc, ("attn", b0), NullCtx(), early_stop=True)
            gate(f"early-stop sweep: intact stats entering block {b0} [{t}] equal RB-8's fit bit for bit", key,
                 n == N_FIT_MEAN and np.array_equal(mu, RB8[key]["fit"][t][f"mu_b{b0}"]) and np.array_equal(Vp, RB8[key]["fit"][t][f"Vp_b{b0}"]),
                 f"max |delta mu| {np.abs(mu - RB8[key]['fit'][t][f'mu_b{b0}']).max():.1e}")
            sub = fc[: max(1, len(fc) // 8)]
            a1 = site_stats(M, sub, ("mlp", N_BLOCKS - 1), NullCtx(), early_stop=True)
            a2 = site_stats(M, sub, ("mlp", N_BLOCKS - 1), NullCtx(), early_stop=False)
            gate(f"early-stop sweep equals the repo's predict() sweep at the last MLP input [{t}] (bit for bit)", key,
                 np.array_equal(a1[0], a2[0]) and np.array_equal(a1[1], a2[1]) and a1[2] == a2[2], f"{a1[2]} images")
            prms, save = {}, {}
            for i, s in enumerate(sites):
                ctx = AllSites(model, source, sites[:i], "mean", prms, npf) if i else NullCtx()
                mu, Vp, n, cos = site_stats(M, fc, s, ctx)
                save[f"mu_{site_name(s)}"], save[f"Vp_{site_name(s)}"], save[f"cos_{site_name(s)}"] = mu, Vp, np.array(cos)
                prms[s] = params_to(derive_params(mu, Vp, site=s[1]), DEVICE)
            tmp = f.with_name(f.stem + ".tmp.npz"); np.savez(tmp, **save); os.replace(tmp, f)
            R["fit_sha256"][t] = sha256_file(f)
            print(f"    fit [{t}] mean: {len(sites)} sites in {time.time() - t0:.0f}s", flush=True)
        else:
            gate(f"saved fit [{t}] is the one this result file was built from (sha256)", key, True, R["fit_sha256"][t][:16] + "...")
        M["prm"][t] = params_from_file(f, sites)
        z = np.load(f); rows = []
        for s in sites:
            p = derive_params(z[f"mu_{site_name(s)}"], z[f"Vp_{site_name(s)}"], site=s[1])
            rows.append({"site": site_name(s), "positional_norm_mean": float(p["c"].norm(dim=1).mean()), "split_half_cos": float(z[f"cos_{site_name(s)}"])})
        R["edit_params"][t] = rows
        if not done:
            if MODEL_SPECS[key]["family"] == "rope":
                gate(f"RoPE: the first edited block input (block 1) is fitted after mlp0, so it differs from RB-8's [{t}]", key,
                     not np.array_equal(z["mu_attn1"], RB8[key]["fit"][t]["mu_b1"]))
            else:
                gate(f"first site attn0 is fitted on the intact model, so it equals RB-8's block-0 fit bit for bit [{t}]", key,
                     np.array_equal(z["mu_attn0"], RB8[key]["fit"][t]["mu_b0"]))
            for s in [x for x in sites if x in (("attn", SITE_MAIN), ("mlp", SITE_MAIN), ("mlp", N_BLOCKS - 1), ("out", N_BLOCKS - 1))]:
                upto = sites[: sites.index(s) + 1]
                mu, _, _, _ = site_stats(M, fc, s, AllSites(model, source, upto, "mean", M["prm"][t], npf))
                m0 = z[f"mu_{site_name(s)}"]; c0 = m0 - m0.mean(0); c1 = mu - mu.mean(0); rms = float(np.sqrt((m0 ** 2).mean()))
                v = float(np.sqrt((c1 ** 2).mean())) / rms
                gate(f"sequential edit flattens the per-slot mean at {site_name(s)} on the fitting images [{t}]", key, v <= FLAT_TOL,
                     f"leftover / token rms {v:.2e}; positional component / token rms {float(np.sqrt((c0 ** 2).mean())) / rms:.2e}")
        del fc


## Gates on the saved results
All read the saved file, so a resumed or re-loaded run still checks every arm. "Bit for bit" = identical values.

In [ ]:
def _maxdiff(a, b): return max((abs(x - y) for x, y in zip(a, b)), default=0.0)
C_NAME = {"baseline": "baseline", "floor": "floor", "ape_floor": "ape_off", "both_floor": "both_off"}

def check_refs(key, R, P):
    ok = []; s = R["ssdc"]; fam = MODEL_SPECS[key]["family"]; cls = MODEL_SPECS[key]["classifier"]
    R8 = RB8[key]["R"]
    if fam != "ape":
        ca = RB0[key]["A"]["conditions"]
        for n in ["baseline"] + floors_of(key):
            d = _maxdiff(s[n]["ssdc"][:13], ca[n]["ssdc"])
            ok.append(gate(f"{n} SSDC equals RB-0 part A bit for bit (13 readouts)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}"))
        if cls:
            cc, Pc = RB0[key]["C"]["conditions"], RB0[key]["P"]
            for n in ["baseline"] + floors_of(key):
                good = R["acc"][n]["acc"] == cc[C_NAME[n]]["acc"] and np.array_equal(P[n], Pc[C_NAME[n]])
                ok.append(gate(f"{n} top-1 equals RB-0 part C '{C_NAME[n]}' bit for bit (mean and per image)", key, good,
                               f"{R['acc'][n]['acc'] * 100:.2f} vs {cc[C_NAME[n]]['acc'] * 100:.2f}"))
    else:
        if key in W1B:
            for n, k1 in (("baseline", "baseline"), ("floor", "floor")):
                d = _maxdiff(s[n]["ssdc"][:N_BLOCKS], W1B[key][k1])
                ok.append(gate(f"{n} SSDC agrees with W1b (repack, 1,000 images; another session) within {W1B_TOL_SSDC}", key,
                               d <= W1B_TOL_SSDC, f"max |delta| {d:.1e}"))
        if key in W7A:
            w = W7A[key]; n7 = len(w["img"]); ours = P["baseline"][:n7]
            a7, ao = w["acc"] * 100, float(ours.mean() * 100); agree = float((ours == w["img"]).mean())
            ok.append(gate(f"intact top-1 agrees with W7 on the same first {n7} ILSVRC images (another session)", key,
                           abs(ao - a7) <= W7_TOL_POINTS and agree >= W7_MIN_AGREE, f"{ao:.2f} vs {a7:.2f}; per-image agreement {agree * 100:.2f}%"))
    for n in ["baseline"] + floors_of(key):
        d = _maxdiff(s[n]["ssdc"][:13], R8["ssdc"][n]["ssdc"])
        ok.append(gate(f"{n} SSDC equals RB-8 bit for bit (13 readouts)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}"))
        if cls:
            ok.append(gate(f"{n} top-1 equals RB-8 bit for bit (mean and per image)", key,
                           R["acc"][n]["acc"] == R8["acc"][n]["acc"] and np.array_equal(P[n], RB8[key]["P"][n])))
    pb, p8 = s["baseline"].get("probes", {}), R8["ssdc"]["baseline"].get("probes", {})
    ok.append(gate("intact probes at the final readout equal RB-8's (same code, seeds and images)", key,
                   str(FINAL) in pb and str(FINAL) in p8 and all(pb[str(FINAL)][k] == p8[str(FINAL)][k] for k in ("exact", "row", "col"))))
    d = _maxdiff(s["rb8_mean"]["ssdc"][:13], R8["ssdc"]["all_mean"]["ssdc"])
    ok.append(gate("RB-8's all-block mean erasure, rebuilt from RB-8's fit files, equals RB-8 bit for bit (SSDC, 13 readouts)", key,
                   d <= EXACT_TOL, f"max |delta| {d:.1e}"))
    if cls:
        good = R["acc"]["rb8_mean"]["acc"] == R8["acc"]["all_mean"]["acc"] and np.array_equal(P["rb8_mean"], RB8[key]["P"]["all_mean"])
        ok.append(gate("RB-8's all-block mean erasure equals RB-8 bit for bit (top-1, mean and per image)", key, good,
                       f"{R['acc']['rb8_mean']['acc'] * 100:.2f} vs {R8['acc']['all_mean']['acc'] * 100:.2f}"))
    return all(ok)

def check_arms(key, R, P):
    arms = plan(key); s = R["ssdc"]
    miss = [f"{k}:{n}" for (k, n) in arms if n not in R[k]]
    gate("every planned arm measured", key, not miss, f"{len(arms) - len(miss)}/{len(arms)}" + (f"; missing {miss[:4]}" if miss else ""))
    badh = [f"{k}:{n}" for (k, n), (spec, _) in arms.items() if n in R[k] and R[k][n]["hooks"] != expected_hooks(key, spec)]
    gate("every arm registered exactly the expected hooks", key, not badh, ",".join(badh[:5]))
    badn = [f"{k}:{n}" for (k, n) in arms if n in R[k] and R[k][n]["n_images"] != (N_ACC if k == "acc" else N_SSDC)]
    gate("every arm saw every image", key, not badn, ",".join(badn[:5]))
    badp = [n for (k, n), (_, ro) in arms.items() if k == "ssdc" and n in s and sorted(map(int, s[n].get("probes", {}))) != sorted(ro)]
    gate("every arm has its planned probes", key, not badp, ",".join(badp[:5]))
    badsw = [n for n in s if not s[n].get("sweeps_identical", True)]
    gate("arms probed in several sweeps gave the same SSDC in every sweep (bit for bit)", key, not badsw, ",".join(badsw))
    base = s["baseline"]["ssdc"]
    if "full_mean" in s:
        f_ = s["full_mean"]["ssdc"]
        ch = [site_name(x) for x in sites_of(key) if site_readout(x) != FINAL and f_[site_readout(x)] == base[site_readout(x)]]
        gate("full_mean changes the stream at every edited site's readout", key, not ch, ",".join(ch[:5]))
        if MODEL_SPECS[key]["family"] == "rope":
            gate("RoPE: readout 0 (block 0's attention input) is untouched by every site arm", key,
                 all(s[n]["ssdc"][0] == base[0] for n in ("full_mean", "full_offset") if n in s))
    if not TEST_MODE:
        ses = R.get("sessions", [])
        gate("every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002", key,
             bool(ses) and all(x.get("cudnn") == EXPECT_CUDNN and x.get("torch") == EXPECT_TORCH for x in ses), f"{len(ses)} session(s)")


## Measurement
Per model: the sequential fit, then intact, floor(s) and RB-8's arm and their gates (a failure stops that model when
`RB9_STRICT=1`), then `full_mean` and `full_offset`. Saves after every arm; a re-run continues where it stopped.

In [ ]:
def run_model(M):
    key = M["key"]
    R, P = load_R(key)
    if R is None:
        R, P = {"model": key, "name": MODEL_SPECS[key]["name"], "family": MODEL_SPECS[key]["family"], "notebook": "RB-9", "config": CFG_FP,
                "n_heads": M["n_heads"], "n_prefix": M["n_prefix"], "sites": [site_name(s) for s in sites_of(key)],
                "acc": {}, "ssdc": {}, "sessions": []}, {}
    else:
        print(f"  resuming {model_file(key).name}: {len(R['acc'])} top-1 arms, {len(R['ssdc'])} SSDC arms saved")
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff")})
    run_fit(M, R); save_R(key, R, P)
    fit_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if fit_fail and STRICT and not TEST_MODE:
        gate("arms measured", key, False, "STOPPED: a setup or fit gate failed (RB9_STRICT=1). Send the gate report."); return R
    arms = plan(key); t0 = time.time(); n_new = [0]
    def run(k, name):
        spec, ro = arms[(k, name)]
        if name in R[k]: return
        if k == "acc":
            a, c, hooks = accuracy(M, build_ctx(M, spec, k)); R["acc"][name] = {"acc": a, "hooks": hooks, "n_images": int(len(c)), "spec": list(spec)}; P[name] = c
        else:
            r = measure_with_probes(M, lambda: build_ctx(M, spec, k), ro)
            R["ssdc"][name] = {"ssdc": r["ssdc"], "hooks": r["hooks"], "n_images": r["n_images"], "spec": list(spec), "probes": r["probes"],
                               "sweeps": r["sweeps"], "sweeps_identical": r["sweeps_identical"]}
        n_new[0] += 1; save_R(key, R, P)
    refs = [(k, n) for (k, n) in arms if n in ["baseline", "rb8_mean"] + floors_of(key)]
    for k, n in refs: run(k, n)
    if not check_refs(key, R, P) and STRICT and not TEST_MODE:
        gate("arms measured", key, False, "STOPPED: an intact / floor / RB-0 / RB-8 anchor gate failed (RB9_STRICT=1). Send the gate report.")
        return R
    todo = [(k, n) for (k, n) in arms if (k, n) not in refs and n not in R[k]]
    print(f"\n--- {key}: {len(todo)} arms to measure ---", flush=True)
    for i, (k, n) in enumerate(todo):
        run(k, n)
        el = time.time() - t0
        print(f"    {i + 1}/{len(todo)} {k}:{n} | {el / 60:.1f} min", flush=True)
    R["complete"] = all(n in R[k] for (k, n) in arms)
    save_R(key, R, P)
    check_arms(key, R, P)
    gate("model finished", key, True, f"{n_new[0]} new arms, {(time.time() - t0) / 60:.1f} min; complete {R['complete']}")
    return R


In [ ]:
T_RUN = time.time()
for key in MODELS_RUN:
    if MODEL_SPECS[key]["family"] != "ape" and key not in RB0:
        gate("model measured", key, False, "no RB-0 files for this model"); continue
    R0, P0 = load_R(key)
    if R0 and R0.get("complete"):
        print(f"\n{key}: already complete in {model_file(key).name}; loading and re-checking")
        RESULTS[key] = R0; check_refs(key, R0, P0); check_arms(key, R0, P0); continue
    try:
        M = setup_model(key)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", key, False, repr(ex)[:200]); continue
    gate("model setup completed", key, True)
    setup_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if setup_fail and STRICT and not TEST_MODE:
        gate("model measured", key, False, f"STOPPED: {len(setup_fail)} setup gate(s) failed: " + "; ".join(g["gate"] for g in setup_fail)[:300])
        del M; continue
    try:
        RESULTS[key] = run_model(M)
    except SystemExit:
        raise
    except Exception as ex:
        traceback.print_exc(); gate("model measured", key, False, repr(ex)[:200])
    del M
    if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"\nmeasurement finished in {(time.time() - T_RUN) / 60:.1f} min")

## Analysis
Reads only saved files. The reading, the outcome label and the report (`rb9_report.md`) follow the rules in the CONFIG cell.

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

def paired(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)
def show(title, obj):
    print("\n" + title); print("-" * len(title))
    print(obj.to_string(index=False) if isinstance(obj, pd.DataFrame) and len(obj) else obj)
def norm_curve(v, base, flo):
    return [((v[b] - flo[b]) / (base[b] - flo[b])) if (base[b] - flo[b]) >= NORM_MIN_RANGE else None for b in range(N_READOUTS)]
def holds(Nb, pr):
    if pr is None: return None
    return bool((Nb is None or Nb <= ERASE_N_MAX) and pr["exact"] <= ERASE_EXACT_X * pr["chance"]["exact"]
                and pr["row"] <= ERASE_AXIS_X * pr["chance"]["row"] and pr["col"] <= ERASE_AXIS_X * pr["chance"]["col"])
def f2(x, n=2): return "—" if x is None else f"{x:.{n}f}".replace("-", "−")

print(RULE_ERASE); print(RULE_NORM); print(RULE_TOP1); print(RULE_READING); print(RULE_OUTCOME)
ANALYSIS, TABLES = {}, {"reading": [], "sites": [], "curves": [], "summary": []}
for key in MODELS_RUN:
    R = RESULTS.get(key) or (json.loads(model_file(key).read_text()) if model_file(key).exists() else None)
    if not R or not R.get("complete"):
        print(f"{key}: not complete; left out of the analysis"); continue
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    s_ = R["ssdc"]; base = s_["baseline"]["ssdc"]; flo = s_[floor_name(key)]["ssdc"]
    p8 = RB8[key]["R"]["ssdc"]["baseline"].get("probes", {})
    intact_probe = {**{int(b): v for b, v in p8.items()}, **{int(b): v for b, v in s_["baseline"].get("probes", {}).items()}}
    for name, arm in s_.items():
        Nc = norm_curve(arm["ssdc"], base, flo)
        for r in range(N_READOUTS):
            TABLES["curves"].append({"model": key, "arm": name, "readout": readout_label(r), "ssdc": arm["ssdc"][r], "N": Nc[r]})
    fm = s_["full_mean"]; Nf = norm_curve(fm["ssdc"], base, flo); N8 = norm_curve(s_["rb8_mean"]["ssdc"], base, flo)
    rows = []
    for st in sites_of(key):
        r = site_readout(st); pr = fm["probes"].get(str(r)); ip = intact_probe.get(r)
        h = holds(Nf[r], pr)
        rows.append({"model": key, "site": site_name(st), "readout": readout_label(r), "N": Nf[r], "N_intact_ssdc": base[r],
                     "exact": pr["exact"], "row": pr["row"], "col": pr["col"], "exact_intact": ip["exact"] if ip else None,
                     "exact_x_chance": pr["exact"] / pr["chance"]["exact"], "row_x_chance": pr["row"] / pr["chance"]["row"],
                     "col_x_chance": pr["col"] / pr["chance"]["col"], "holds": h})
    TABLES["sites"] += rows
    ex = [x["exact"] for x in rows]; axx = [max(x["row_x_chance"], x["col_x_chance"]) for x in rows]
    exi = [x["exact_intact"] for x in rows if x["exact_intact"] is not None]
    out = {"family": MODEL_SPECS[key]["family"], "label": MODEL_SPECS[key]["label"], "sites": len(rows),
           "sites_holding": sum(1 for x in rows if x["holds"]), "exact_min": min(ex), "exact_max": max(ex), "axis_x_max": max(axx),
           "exact_intact_max": max(exi) if exi else None,
           "N_b11_full": Nf[11], "N_mlp11_full": Nf[MLP0 + 11], "N_final_full": Nf[FINAL], "N_b11_rb8": N8[11], "N_final_rb8": N8[FINAL]}
    cls = MODEL_SPECS[key]["classifier"]
    if cls:
        F_, _ = paired(P["baseline"], P[floor_name(key)])
        C, Cse = paired(P["baseline"], P["full_mean"]); C0, C0se = paired(P["baseline"], P["full_offset"])
        d, dse = paired(P["full_offset"], P["full_mean"]); C8, C8se = paired(P["baseline"], P["rb8_mean"])
        d8, d8se = paired(P["rb8_mean"], P["full_mean"])
        v = "NOT READABLE (F <= 0)" if F_ <= 0 else ("BARELY USED" if C < READ_SHARE * F_ else ("USED" if d > READ_SE * dse else "CANNOT TELL"))
        out["reading"] = {"model": key, "C": C, "C_se": Cse, "C0": C0, "C0_se": C0se, "C_minus_C0": d, "diff_se": dse, "F": F_,
                          "share_of_F": C / F_ if F_ > 0 else None, "C_rb8": C8, "C_rb8_se": C8se, "C_minus_C_rb8": d8, "C_minus_C_rb8_se": d8se,
                          "top1_intact": float(P["baseline"].mean() * 100), "verdict": v}
        TABLES["reading"].append(out["reading"])
    TABLES["summary"].append({"model": key, **{k: v for k, v in out.items() if k != "reading"}})
    ANALYSIS[key] = out

rope_cls = [k for k in ("naver", "small", "reg1_gap") if k in ANALYSIS and "reading" in ANALYSIS[k]]
verd = {k: ANALYSIS[k]["reading"]["verdict"] for k in rope_cls}
if len(rope_cls) < 3: OUTCOME = "NOT READABLE (a RoPE classifier is missing)"
elif all(v == "BARELY USED" for v in verd.values()): OUTCOME = "A"
elif any(v == "USED" for v in verd.values()): OUTCOME = "B"
else: OUTCOME = "C"
print("\nOUTCOME:", OUTCOME, verd)
if TABLES["reading"]: show("Reading (rule above)", pd.DataFrame(TABLES["reading"]).round(3))
if TABLES["summary"]: show("Per model: erasure at every site", pd.DataFrame(TABLES["summary"]).round(4))
if TABLES["sites"]:
    sd = pd.DataFrame(TABLES["sites"]); show("Every edited site (full_mean)", sd.round(4))
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb9_{name}.csv", index=False)

# report
def num(x, n=2): return "\u2014" if x is None else f"{x:.{n}f}".replace("-", "\u2212")
def rng(vals, n=2):
    vals = [v for v in vals if v is not None]
    if not vals: return "\u2014"
    lo, hi = min(vals), max(vals)
    return num(lo, n) if num(lo, n) == num(hi, n) else (f"{num(lo, n)}\u2013{num(hi, n)}" if lo >= 0 else f"{num(lo, n)} to {num(hi, n)}")
L = [f"# RB-9 results — generated {RUN_STAMP}\n"]
gd = [g for g in GATES]
L.append("Rules (fixed before the run):\n")
for r_ in (RULE_ERASE, RULE_NORM, RULE_READING, RULE_OUTCOME): L.append(f"- {r_}")
_verd_txt = ", ".join(MODEL_SPECS[k]["label"] + ": " + v for k, v in verd.items())
L.append(f"\n**Outcome: {OUTCOME}** ({_verd_txt})\n")
L.append("## Top-1 (points ± paired SE, 5,000 ILSVRC images)\n")
L.append("| model | intact | F (full removal) | RB-8 edit (block inputs only) | RB-9 edit (every site) | % of F | control (wrong slots) | RB-9 − RB-8 | verdict |")
L.append("|---|---|---|---|---|---|---|---|---|")
for r_ in TABLES["reading"]:
    L.append(f"| {MODEL_SPECS[r_['model']]['label']} | {num(r_['top1_intact'])} | {num(r_['F'])} | {num(r_['C_rb8'])} ± {num(r_['C_rb8_se'])} | "
             f"{num(r_['C'])} ± {num(r_['C_se'])} | {num(None if r_['share_of_F'] is None else 100 * r_['share_of_F'], 1)}% | {num(r_['C0'])} ± {num(r_['C0_se'])} | "
             f"{num(r_['C_minus_C_rb8'])} ± {num(r_['C_minus_C_rb8_se'])} | {r_['verdict']} |")
L.append("\n## The erasure at every site (normalised SSDC; probes on held-out images, chance 0.51% exact)\n")
L.append("| model | sites | sites meeting the erasure check | exact probe over the sites | intact exact probe, max over the same readouts | N entering block 11: RB-8 → RB-9 | N at MLP 11 (RB-9) | N at the final output: RB-8 → RB-9 |")
L.append("|---|---|---|---|---|---|---|---|")
for r_ in TABLES["summary"]:
    ei = "\u2014" if r_["exact_intact_max"] is None else f"{100 * r_['exact_intact_max']:.1f}%"
    L.append(f"| {r_['label']} | {r_['sites']} | {r_['sites_holding']} | {100 * r_['exact_min']:.1f}\u2013{100 * r_['exact_max']:.1f}% | {ei} | "
             f"{f2(r_['N_b11_rb8'])} \u2192 {f2(r_['N_b11_full'])} | {f2(r_['N_mlp11_full'])} | {f2(r_['N_final_rb8'])} \u2192 {f2(r_['N_final_full'])} |")
rr = {k: ANALYSIS[k]["reading"] for k in rope_cls}
ape = [ANALYSIS[k]["reading"] for k in ("supervised", "augreg", "sam", "deit3") if k in ANALYSIS and "reading" in ANALYSIS[k]]
L.append("\n## Text for the paper (by the outcome above)\n")
if OUTCOME == "A":
    cr = rng([v["C"] for v in rr.values()], 1); sr = rng([100 * v["share_of_F"] for v in rr.values() if v["share_of_F"] is not None], 1)
    ar = rng([v["C"] for v in ape], 1) if ape else "—"; asr = rng([100 * v["share_of_F"] for v in ape if v["share_of_F"] is not None], 0) if ape else "—"
    L.append(f"- M17d, last sentence: “Subtracting it before the attention and before the MLP of every block at once costs the RoPE classifiers "
             f"{cr} top-1 points ({sr}% of removing every rotation) and APE {ar} ({asr}% of removing the embedding); it removes most decodable "
             f"position but not the RoPE classifiers’ SSDC (Appendix G.3).”")
    L.append("- G.3, the paragraph on the erasure at every block: describe the RB-9 edit (both sites of every block, plus the last block’s output) "
             "with the numbers in the two tables above, and add: “Editing only the input of each block, which leaves the position that attention "
             f"writes within a block visible to that block’s MLP, gives the same result (RoPE classifiers {rng([v['C_rb8'] for v in rr.values()], 2)} points).”")
    L.append("- Table 27: replace the cost and probe columns with RB-9’s, keep RB-8’s cost as one extra column.")
elif OUTCOME == "B":
    L.append("- At least one RoPE classifier uses the position that attention writes into the stream within a block. M17d’s last sentence and G.3’s "
             "conclusion (“barely use the position their residual stream carries”) must be narrowed to the block inputs (RB-8), and the RB-9 cost "
             "reported next to it. The four of us decide the wording before anything goes into the draft.")
else:
    L.append("- No general statement. Report the per-model numbers in G.3; M17d keeps RB-8’s sentence with “entering every block”. The four of us decide.")
L.append("\nGates: every gate of every session is in rb9_gate_report.csv (written by the last cell); the run is valid only if it shows 0 FAIL and 0 SKIP.")
(OUT_DIR / "rb9_report.md").write_text("\n".join(L))
print("\n".join(L))


## Gate report and export

In [ ]:
_all = []
for f in sorted(OUT_DIR.glob("rb9_gates_session_*.json")): _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT  ({len(set(g['session'] for g in _all))} session(s), {len(_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
complete = {k: bool((RESULTS.get(k) or {}).get("complete")) or (model_file(k).exists() and json.loads(model_file(k).read_text()).get("complete", False)) for k in MODELS_RUN}
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {n_skip} skipped | complete {complete}")
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in gdf[gdf.status == "FAIL"].iterrows(): print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)
export = {"notebook": "RB-9", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "environment": ENV, "repo": str(repo), "repo_commit": REPO_COMMIT,
          "rb0_dir": str(_rb0) if _rb0 else None, "config": {**CFG_FP, "models": MODELS_RUN, "strict": STRICT},
          "rules": {"erasure": RULE_ERASE, "normalisation": RULE_NORM, "top1": RULE_TOP1, "reading": RULE_READING, "outcome": RULE_OUTCOME}, "outcome": OUTCOME,
          "analysis": ANALYSIS, "gates": gdf.to_dict("records"),
          "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "complete": complete, "sessions": sorted(set(g["session"] for g in _all))}}
(OUT_DIR / f"rb9_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "rb9_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb9_{name}.csv", index=False)
import zipfile
archive = str(OUT_DIR) + ".zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z_:
    for p in sorted(OUT_DIR.iterdir()):
        if not p.name.startswith("rb9_fit_"): z_.write(p, f"{OUT_DIR.name}/{p.name}")
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")